In [1]:
import os

# Make only physical GPU 1 visible
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("CUDA_VISIBLE_DEVICES set to 1")

CUDA_VISIBLE_DEVICES set to 1. Restart the kernel now.


In [2]:
import torch, os
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("device_count:", torch.cuda.device_count())
print("gpu0 name:", torch.cuda.get_device_name(0))

CUDA_VISIBLE_DEVICES: 1
device_count: 1
gpu0 name: NVIDIA RTX A6000


In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import sys
import math
import json
import time
import random
import argparse
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm import HyperbolicLCM


# CONFIG

@dataclass
class FinetuneConfig:
    datasets_to_run: Tuple[str, ...] = ("anatomy",)

    cache_dir: str = "mcq_cache"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 64

    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    epochs: int = 3
    train_batch_size: int = 4
    eval_batch_size: int = 8
    grad_accum_steps: int = 1
    lr: float = 0.0
    head_lr: float = 3e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    max_grad_norm: float = 1.0
    label_smoothing: float = 0.10

    freeze_encoder: bool = True
    freeze_hlcm: bool = True

    head_dropout: float = 0.50

    seed: int = 42
    num_workers: int = 4
    use_bf16: bool = True
    prefer_gpu_index: int = 0

    out_dir: str = "runs/mcq_hlcm_mmlu_auxtrain"

    aux_train_config_name: str = "auxiliary_train"
    aux_train_split_name: str = "train"

    subject_val_split_name: str = "validation"
    subject_test_split_name: str = "test"


def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds: float) -> str:
    seconds = max(float(seconds), 0.0)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def answerkey_to_index(answer_key: Any, num_choices: int) -> int:
    if isinstance(answer_key, int):
        idx = int(answer_key)
    else:
        answer_key = str(answer_key).strip()
        alpha = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
        numeric = {"0": 0, "1": 1, "2": 2, "3": 3, "4": 4}

        if answer_key in alpha:
            idx = alpha[answer_key]
        elif answer_key in numeric:
            idx = numeric[answer_key]
        else:
            raise ValueError(f"Unknown answer label: {answer_key}")

    if idx >= num_choices:
        raise ValueError(f"answer={answer_key} -> idx={idx}, but num_choices={num_choices}")
    return idx


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None
    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


def resolve_existing_split(ds_dict, preferred_name: str, fallback_names: List[str]) -> str:
    if preferred_name in ds_dict:
        return preferred_name
    for name in fallback_names:
        if name in ds_dict:
            return name
    raise KeyError(f"Could not find split '{preferred_name}'. Available: {list(ds_dict.keys())}")


def masked_choice_cross_entropy(
    logits: torch.Tensor,
    labels: torch.Tensor,
    choice_mask: torch.Tensor,
    label_smoothing: float = 0.0,
) -> torch.Tensor:
    b, _ = logits.shape
    log_probs = F.log_softmax(logits, dim=-1)

    losses = []
    for i in range(b):
        valid = choice_mask[i]
        valid_idx = torch.nonzero(valid, as_tuple=False).squeeze(-1)
        lp = log_probs[i, valid]

        gold_global = int(labels[i].item())
        local_pos = (valid_idx == gold_global).nonzero(as_tuple=False)
        if local_pos.numel() == 0:
            raise RuntimeError(f"Gold label {gold_global} not found among valid choices in row {i}")
        local_pos = int(local_pos.item())

        cv = int(lp.size(0))
        if label_smoothing > 0.0:
            target = torch.full_like(lp, fill_value=label_smoothing / cv)
            target[local_pos] += (1.0 - label_smoothing)
            loss_i = -(target * lp).sum()
        else:
            loss_i = -lp[local_pos]

        losses.append(loss_i)

    return torch.stack(losses).mean()


# CONCEPTIZER

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.encoder = AutoModel.from_pretrained(model_name).to(device)
        self.encoder.eval()

        for p in self.encoder.parameters():
            p.requires_grad = False

        self.embed_dim = int(self.encoder.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.embed_dim, dtype=torch.float32)

        inputs = self.tokenizer(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device, non_blocking=True) for k, v in inputs.items()}

        use_amp = (self.device.type == "cuda")
        if self.device.type == "cuda":
            dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
            with amp.autocast(device_type="cuda", enabled=use_amp, dtype=dtype):
                out = self.encoder(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.encoder(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _text_to_chunk_texts(self, text: str) -> List[str]:
        ids = self.tokenizer(text, add_special_tokens=False)["input_ids"]
        if len(ids) == 0:
            return []

        chunk_texts = []
        for i in range(0, len(ids), self.chunk_tok_len):
            chunk_ids = ids[i:i + self.chunk_tok_len]
            chunk_text = self.tokenizer.decode(chunk_ids, clean_up_tokenization_spaces=True)
            chunk_texts.append(chunk_text)
        return chunk_texts

    def encode_text(self, text: str) -> torch.Tensor:
        chunk_texts = self._text_to_chunk_texts(text)
        if len(chunk_texts) == 0:
            return torch.zeros(self.seq_len, self.embed_dim, dtype=torch.float32)

        if len(chunk_texts) > self.seq_len:
            chunk_texts = chunk_texts[:self.seq_len]

        out = []
        for i in range(0, len(chunk_texts), self.batch_size):
            batch = chunk_texts[i:i + self.batch_size]
            out.append(self._embed_chunk_texts(batch))

        embs = torch.cat(out, dim=0)
        t = embs.size(0)

        if t < self.seq_len:
            pad = torch.zeros(self.seq_len - t, embs.size(1), dtype=embs.dtype)
            embs = torch.cat([embs, pad], dim=0)

        return embs[:self.seq_len]

    def encode_choice_set(self, question_stem: str, choice_texts: List[str]) -> torch.Tensor:
        feats = []
        for ch in choice_texts:
            text = f"Question: {question_stem}\nAnswer Choice: {ch}"
            feats.append(self.encode_text(text))
        return torch.stack(feats, dim=0)


# DATA PREP

def load_mmlu_auxiliary_train():
    return load_dataset("cais/mmlu", "auxiliary_train")


def load_mmlu_subject(subject_name: str):
    return load_dataset("cais/mmlu", subject_name)


def unwrap_mmlu_row(example: Dict[str, Any]) -> Dict[str, Any]:
    if isinstance(example, dict) and len(example) == 1:
        only_key = next(iter(example.keys()))
        only_val = example[only_key]
        if only_key in {"train", "validation", "val", "test", "dev"} and isinstance(only_val, dict):
            return only_val
    return example


def normalize_choices_field(example: Dict[str, Any]) -> Tuple[str, List[str], int]:
    example = unwrap_mmlu_row(example)

    # question
    if "question" in example:
        stem = str(example["question"])
    elif "input" in example:
        stem = str(example["input"])
    elif "prompt" in example:
        stem = str(example["prompt"])
    else:
        raise KeyError(f"question-like field not found. keys={list(example.keys())}")

    # choices
    if "choices" in example:
        raw_choices = example["choices"]

        if isinstance(raw_choices, list):
            choice_texts = [str(x) for x in raw_choices]
        elif isinstance(raw_choices, dict):
            if "text" in raw_choices:
                choice_texts = [str(x) for x in raw_choices["text"]]
            elif "label" in raw_choices:
                choice_texts = [str(x) for x in raw_choices["label"]]
            else:
                raise KeyError(f"choices present but unsupported structure. keys={list(raw_choices.keys())}")
        else:
            raise TypeError(f"Unsupported type for choices: {type(raw_choices)}")

    elif all(k in example for k in ["A", "B", "C", "D"]):
        choice_texts = [
            str(example["A"]),
            str(example["B"]),
            str(example["C"]),
            str(example["D"]),
        ]
        if "E" in example:
            choice_texts.append(str(example["E"]))

    elif "options" in example:
        raw_choices = example["options"]
        if isinstance(raw_choices, list):
            choice_texts = [str(x) for x in raw_choices]
        else:
            raise TypeError(f"Unsupported type for options: {type(raw_choices)}")

    else:
        raise KeyError(f"choice field not found. keys={list(example.keys())}")

    # answer
    if "answer" in example:
        answer_value = example["answer"]
    elif "answerKey" in example:
        answer_value = example["answerKey"]
    elif "target" in example:
        answer_value = example["target"]
    elif "label" in example and not isinstance(example["label"], list):
        answer_value = example["label"]
    else:
        raise KeyError(f"answer field not found. keys={list(example.keys())}")

    label = answerkey_to_index(answer_value, len(choice_texts))
    return stem, choice_texts, label


def build_or_load_cached_split(
    cache_name_prefix: str,
    split_name: str,
    split_data,
    conceptizer: DebertaConceptizer,
    cache_dir: str,
) -> List[Dict[str, Any]]:
    ensure_dir(cache_dir)
    safe_ds = cache_name_prefix.replace("/", "_")
    path = os.path.join(
        cache_dir,
        f"{safe_ds}_{split_name}_seq{conceptizer.seq_len}_tok{conceptizer.chunk_tok_len}.pt"
    )

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    rows = []
    skipped = 0
    print(f"[cache] building {cache_name_prefix} / {split_name}")

    for ex in tqdm(split_data, desc=f"Conceptizing {cache_name_prefix}-{split_name}"):
        try:
            ex = unwrap_mmlu_row(ex)
            stem, choice_texts, label = normalize_choices_field(ex)
            x = conceptizer.encode_choice_set(stem, choice_texts)
            choice_mask = torch.ones(x.size(0), dtype=torch.bool)

            rows.append({
                "x": x,
                "label": int(label),
                "choice_mask": choice_mask,
                "num_choices": int(x.size(0)),
            })
        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example in {cache_name_prefix}-{split_name}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")
    return rows


class MCQFeatureDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx: int):
        r = self.rows[idx]
        return {
            "x": r["x"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "choice_mask": r["choice_mask"],
            "num_choices": r["num_choices"],
        }


def mcq_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    max_c = max(item["x"].size(0) for item in batch)
    t = batch[0]["x"].size(1)
    d = batch[0]["x"].size(2)
    b = len(batch)

    x = torch.zeros(b, max_c, t, d, dtype=torch.float32)
    choice_mask = torch.zeros(b, max_c, dtype=torch.bool)
    labels = torch.zeros(b, dtype=torch.long)

    for i, item in enumerate(batch):
        c = item["x"].size(0)
        x[i, :c] = item["x"]
        choice_mask[i, :c] = item["choice_mask"]
        labels[i] = item["label"]

    return {
        "x": x,
        "choice_mask": choice_mask,
        "labels": labels,
    }


class MCQHead(nn.Module):
    def __init__(self, model: HyperbolicLCM, dropout: float = 0.5):
        super().__init__()
        self.model = model
        dim = model.layers[0].attn.embed_dim

        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dim, 1)

        nn.init.normal_(self.classifier.weight, mean=0.0, std=0.01)
        nn.init.zeros_(self.classifier.bias)

    def forward(
        self,
        x: torch.Tensor,
        choice_mask: torch.Tensor,
        mu: Optional[torch.Tensor] = None,
        sigma: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        b, c, t, din = x.shape
        x = x.reshape(b * c, t, din)

        if mu is not None and sigma is not None:
            x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

        with torch.no_grad():
            h = self.model(x)

        last_h = h[:, -1, :]
        last_tan = self.model.manifold.logmap0(last_h)
        last_tan = self.norm(last_tan)
        last_tan = self.dropout(last_tan)

        logits = self.classifier(last_tan).view(b, c)
        logits = logits.masked_fill(~choice_mask, -1e9)
        return logits


def build_hlcm_from_cfg(cfg: FinetuneConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: FinetuneConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"[load] loaded HLCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    model.eval()
    for p in model.parameters():
        p.requires_grad = False

    return model


class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps: int, total_steps: int, min_lr_scale: float = 0.1):
        self.optimizer = optimizer
        self.warmup_steps = max(0, int(warmup_steps))
        self.total_steps = max(1, int(total_steps))
        self.min_lr_scale = float(min_lr_scale)
        self.step_num = 0
        self.base_lrs = [pg["lr"] for pg in optimizer.param_groups]

    def step(self):
        self.step_num += 1
        s = self.step_num

        if s <= self.warmup_steps and self.warmup_steps > 0:
            scale = s / self.warmup_steps
        else:
            denom = max(1, self.total_steps - self.warmup_steps)
            p = min(1.0, max(0.0, (s - self.warmup_steps) / denom))
            cosine = 0.5 * (1.0 + math.cos(math.pi * p))
            scale = self.min_lr_scale + (1.0 - self.min_lr_scale) * cosine

        for lr0, pg in zip(self.base_lrs, self.optimizer.param_groups):
            pg["lr"] = lr0 * scale


@torch.no_grad()
def evaluate(
    model: MCQHead,
    loader: DataLoader,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> Dict[str, float]:
    model.eval()

    total_loss = 0.0
    total = 0
    correct = 0

    for batch in loader:
        x = batch["x"].to(device, non_blocking=True)
        choice_mask = batch["choice_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        logits = model(x, choice_mask, mu=mu, sigma=sigma)
        loss = masked_choice_cross_entropy(
            logits=logits,
            labels=labels,
            choice_mask=choice_mask,
            label_smoothing=0.0,
        )

        total_loss += float(loss.item()) * x.size(0)
        preds = logits.argmax(dim=-1)
        correct += int((preds == labels).sum().item())
        total += int(x.size(0))

    model.train()
    return {
        "loss": total_loss / max(total, 1),
        "acc": correct / max(total, 1),
    }


def train_one_dataset(subject_name: str, cfg: FinetuneConfig):
    print(f"\nMMLU subject: {subject_name}")
    device = pick_device(cfg.prefer_gpu_index)
    set_seed(cfg.seed)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    out_dir = os.path.join(cfg.out_dir, subject_name.replace("/", "_"))
    ensure_dir(out_dir)
    ensure_dir(cfg.cache_dir)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=device,
    )

    aux_ds = load_mmlu_auxiliary_train()
    aux_train_split = resolve_existing_split(
        aux_ds,
        cfg.aux_train_split_name,
        ["train", "auxiliary_train"]
    )

    first_aux_raw = aux_ds[aux_train_split][0]
    first_aux_unwrapped = unwrap_mmlu_row(first_aux_raw)
    print("[debug] aux train split names:", list(aux_ds.keys()))
    print("[debug] first aux raw keys:", first_aux_raw.keys())
    print("[debug] first aux raw sample:", first_aux_raw)
    print("[debug] first aux unwrapped keys:", first_aux_unwrapped.keys())
    print("[debug] first aux unwrapped sample:", first_aux_unwrapped)

    subject_ds = load_mmlu_subject(subject_name)
    val_split = resolve_existing_split(
        subject_ds,
        cfg.subject_val_split_name,
        ["validation", "val", "dev"]
    )
    test_split = resolve_existing_split(
        subject_ds,
        cfg.subject_test_split_name,
        ["test"]
    )

    print(f"[data] aux config splits: {list(aux_ds.keys())}")
    print(f"[data] subject config splits: {list(subject_ds.keys())}")
    print(f"[data] train <- cais/mmlu/{cfg.aux_train_config_name}:{aux_train_split}")
    print(f"[data] val   <- cais/mmlu/{subject_name}:{val_split}")
    print(f"[data] test  <- cais/mmlu/{subject_name}:{test_split}")

    train_rows = build_or_load_cached_split(
        f"mmlu_auxtrain_for_{subject_name}",
        aux_train_split,
        aux_ds[aux_train_split],
        conceptizer,
        cfg.cache_dir,
    )
    val_rows = build_or_load_cached_split(
        f"mmlu_{subject_name}",
        val_split,
        subject_ds[val_split],
        conceptizer,
        cfg.cache_dir,
    )
    test_rows = build_or_load_cached_split(
        f"mmlu_{subject_name}",
        test_split,
        subject_ds[test_split],
        conceptizer,
        cfg.cache_dir,
    )

    if len(train_rows) == 0:
        raise RuntimeError("No usable training rows from top-level auxiliary_train config.")
    if len(val_rows) == 0:
        raise RuntimeError(f"No usable validation rows for subject {subject_name}.")

    train_ds = MCQFeatureDataset(train_rows)
    val_ds = MCQFeatureDataset(val_rows)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    test_loader = None
    if len(test_rows) > 0:
        test_ds = MCQFeatureDataset(test_rows)
        test_loader = DataLoader(
            test_ds,
            batch_size=cfg.eval_batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=(device.type == "cuda"),
            collate_fn=mcq_collate,
            drop_last=False,
        )

    hlcm = load_pretrained_hlcm(cfg, device)
    model = MCQHead(hlcm, dropout=cfg.head_dropout).to(device)

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        [{"params": trainable_params, "lr": cfg.head_lr, "weight_decay": cfg.weight_decay}]
    )

    steps_per_epoch = math.ceil(len(train_loader) / max(cfg.grad_accum_steps, 1))
    total_steps = max(1, cfg.epochs * steps_per_epoch)
    warmup_steps = int(cfg.warmup_ratio * total_steps)
    scheduler = WarmupCosineScheduler(
        optimizer,
        warmup_steps=warmup_steps,
        total_steps=total_steps,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"
    amp_dtype = torch.bfloat16

    history = []
    best_val_acc = -1.0
    best_path = os.path.join(out_dir, "best_mcq.pt")

    start_time = time.time()

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        running_loss = 0.0
        seen = 0

        pbar = tqdm(train_loader, desc=f"Train {subject_name} epoch {epoch}/{cfg.epochs}", mininterval=1.0)

        for step, batch in enumerate(pbar, start=1):
            x = batch["x"].to(device, non_blocking=True)
            choice_mask = batch["choice_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=amp_dtype):
                    logits = model(x, choice_mask, mu=mu, sigma=sigma)
                    loss = masked_choice_cross_entropy(
                        logits=logits,
                        labels=labels,
                        choice_mask=choice_mask,
                        label_smoothing=float(cfg.label_smoothing),
                    )
            else:
                logits = model(x, choice_mask, mu=mu, sigma=sigma)
                loss = masked_choice_cross_entropy(
                    logits=logits,
                    labels=labels,
                    choice_mask=choice_mask,
                    label_smoothing=float(cfg.label_smoothing),
                )

            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True)
                continue

            (loss / cfg.grad_accum_steps).backward()

            running_loss += float(loss.item()) * x.size(0)
            seen += int(x.size(0))

            if step % cfg.grad_accum_steps == 0 or step == len(train_loader):
                torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            pbar.set_postfix(loss=f"{running_loss / max(seen, 1):.4f}")

        train_loss = running_loss / max(seen, 1)
        val_metrics = evaluate(model, val_loader, device, mu, sigma)

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["acc"],
        }

        if test_loader is not None:
            test_metrics = evaluate(model, test_loader, device, mu, sigma)
            row["test_loss"] = test_metrics["loss"]
            row["test_acc"] = test_metrics["acc"]
        else:
            test_metrics = None

        history.append(row)

        print(
            f"[epoch {epoch}] "
            f"train_loss={train_loss:.4f} "
            f"val_loss={val_metrics['loss']:.4f} "
            f"val_acc={val_metrics['acc']:.4f}"
            + (f" test_acc={test_metrics['acc']:.4f}" if test_metrics is not None else "")
        )

        if val_metrics["acc"] > best_val_acc:
            best_val_acc = float(val_metrics["acc"])
            torch.save(
                {
                    "cfg": asdict(cfg),
                    "subject_name": subject_name,
                    "train_source": f'load_dataset("cais/mmlu", "{cfg.aux_train_config_name}")["{aux_train_split}"]',
                    "eval_source": f'load_dataset("cais/mmlu", "{subject_name}")["{val_split}"]',
                    "test_source": f'load_dataset("cais/mmlu", "{subject_name}")["{test_split}"]',
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "hlcm_state": model.model.state_dict(),
                    "best_val_acc": best_val_acc,
                    "history": history,
                },
                best_path,
            )
            print(f"[save] best checkpoint -> {best_path}")

    wall = time.time() - start_time

    best_obj = torch.load(best_path, map_location=device)
    model.load_state_dict(best_obj["model_state"], strict=True)

    final_val = evaluate(model, val_loader, device, mu, sigma)
    final_test = evaluate(model, test_loader, device, mu, sigma) if test_loader is not None else None

    summary = {
        "subject_name": subject_name,
        "train_source": f'load_dataset("cais/mmlu", "{cfg.aux_train_config_name}")["{aux_train_split}"]',
        "val_source": f'load_dataset("cais/mmlu", "{subject_name}")["{val_split}"]',
        "test_source": f'load_dataset("cais/mmlu", "{subject_name}")["{test_split}"]',
        "best_val_acc": float(final_val["acc"]),
        "best_val_loss": float(final_val["loss"]),
        "test_acc": None if final_test is None else float(final_test["acc"]),
        "test_loss": None if final_test is None else float(final_test["loss"]),
        "epochs": cfg.epochs,
        "wall_time_sec": float(wall),
        "wall_time_hms": fmt_hms(wall),
        "num_train_examples": len(train_ds),
        "num_val_examples": len(val_ds),
        "num_test_examples": len(test_rows),
        "history": history,
    }

    with open(os.path.join(out_dir, "summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print(f"[done] {subject_name}")
    print(json.dumps(summary, indent=2))

    return summary


# MAIN

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--ckpt_path", type=str, default="runs/hyperbolic_cluster/checkpoints/ckpt_best.pt")
    p.add_argument("--normalizer_path", type=str, default="normalizer.pt")
    p.add_argument("--out_dir", type=str, default="runs/mcq_hlcm_mmlu_auxtrain")
    p.add_argument("--cache_dir", type=str, default="mcq_cache")

    p.add_argument("--epochs", type=int, default=3)
    p.add_argument("--train_batch_size", type=int, default=4)
    p.add_argument("--eval_batch_size", type=int, default=8)
    p.add_argument("--grad_accum_steps", type=int, default=1)

    p.add_argument("--lr", type=float, default=0.0)
    p.add_argument("--head_lr", type=float, default=3e-5)
    p.add_argument("--weight_decay", type=float, default=0.01)
    p.add_argument("--warmup_ratio", type=float, default=0.06)
    p.add_argument("--max_grad_norm", type=float, default=1.0)

    p.add_argument("--seq_len", type=int, default=8)
    p.add_argument("--chunk_tok_len", type=int, default=256)
    p.add_argument("--encoder_batch_size", type=int, default=64)

    p.add_argument("--prefer_gpu_index", type=int, default=0)
    p.add_argument("--seed", type=int, default=42)

    p.add_argument("--label_smoothing", type=float, default=0.10)
    p.add_argument("--subject", type=str, default="anatomy")

    if "ipykernel" in sys.modules:
        return p.parse_args(args=[])

    return p.parse_args()


def main():
    args = parse_args()

    cfg = FinetuneConfig(
        datasets_to_run=(args.subject,),
        ckpt_path=args.ckpt_path,
        normalizer_path=args.normalizer_path,
        out_dir=args.out_dir,
        cache_dir=args.cache_dir,
        epochs=args.epochs,
        train_batch_size=args.train_batch_size,
        eval_batch_size=args.eval_batch_size,
        grad_accum_steps=args.grad_accum_steps,
        lr=args.lr,
        head_lr=args.head_lr,
        weight_decay=args.weight_decay,
        warmup_ratio=args.warmup_ratio,
        max_grad_norm=args.max_grad_norm,
        seq_len=args.seq_len,
        chunk_tok_len=args.chunk_tok_len,
        encoder_batch_size=args.encoder_batch_size,
        prefer_gpu_index=args.prefer_gpu_index,
        seed=args.seed,
        label_smoothing=args.label_smoothing,
        freeze_hlcm=True,
    )

    ensure_dir(cfg.out_dir)

    all_summaries = {}
    for subject_name in cfg.datasets_to_run:
        summary = train_one_dataset(subject_name, cfg)
        all_summaries[subject_name] = summary

    with open(os.path.join(cfg.out_dir, "all_results.json"), "w") as f:
        json.dump(all_summaries, f, indent=2)

    print("\nALL DONE")
    print(json.dumps(all_summaries, indent=2))


if __name__ == "__main__":
    main()


==================== MMLU subject: anatomy ====================


/home/user/anaconda3/envs/raat/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


[debug] aux train split names: ['train']
[debug] first aux raw keys: dict_keys(['train'])
[debug] first aux raw sample: {'train': {'answer': 1, 'choices': ['Adams only.', 'Brooks only.', 'Case only.', 'Adams and Brooks'], 'question': "Davis decided to kill Adams. He set out for Adams's house. Before he got there he saw Brooks, who resembled Adams. Thinking that Brooks was Adams, Davis shot at Brooks. The shot missed Brooks but wounded Case, who was some distance away. Davis had not seen Case. In a prosecution under a statute that proscribes any attempt to commit murder, the district attorney should indicate that the intended victim(s) was/were", 'subject': ''}}
[debug] first aux unwrapped keys: dict_keys(['answer', 'choices', 'question', 'subject'])
[debug] first aux unwrapped sample: {'answer': 1, 'choices': ['Adams only.', 'Brooks only.', 'Case only.', 'Adams and Brooks'], 'question': "Davis decided to kill Adams. He set out for Adams's house. Before he got there he saw Brooks, who r

Conceptizing mmlu_auxtrain_for_anatomy-train: 100%|████████████████████████████████████████| 99842/99842 [46:59<00:00, 35.41it/s]


[cache] saved mcq_cache/mmlu_auxtrain_for_anatomy_train_seq8_tok256.pt (99842 examples, skipped=0)
[cache] building mmlu_anatomy / validation


...

[epoch 1] train_loss=1.4923 val_loss=1.3868 val_acc=0.1429 test_acc=0.2296
[save] best checkpoint -> runs/mcq_hlcm_mmlu_auxtrain/anatomy/best_mcq.pt


Train anatomy epoch 2/3: 100%|████████████████████████████████████████████████| 24961/24961 [30:31<00:00, 13.63it/s, loss=1.4049]


[epoch 2] train_loss=1.4049 val_loss=1.3866 val_acc=0.1429 test_acc=0.2667


Train anatomy epoch 3/3: 100%|████████████████████████████████████████████████| 24961/24961 [30:26<00:00, 13.67it/s, loss=1.3955]


[epoch 3] train_loss=1.3955 val_loss=1.3863 val_acc=0.0714 test_acc=0.2815
[done] anatomy
{
  "subject_name": "anatomy",
  "train_source": "load_dataset(\"cais/mmlu\", \"auxiliary_train\")[\"train\"]",
  "val_source": "load_dataset(\"cais/mmlu\", \"anatomy\")[\"validation\"]",
  "test_source": "load_dataset(\"cais/mmlu\", \"anatomy\")[\"test\"]",
  "best_val_acc": 0.14285714285714285,
  "best_val_loss": 1.3867923532213484,
  "test_acc": 0.22962962962962963,
  "test_loss": 1.3864924633944475,
  "epochs": 3,
  "wall_time_sec": 5551.027699232101,
  "wall_time_hms": "01:32:31",
  "num_train_examples": 99842,
  "num_val_examples": 14,
  "num_test_examples": 135,
  "history": [
    {
      "epoch": 1,
      "train_loss": 1.492334360209144,
      "val_loss": 1.3867943797792708,
      "val_acc": 0.14285714285714285,
      "test_loss": 1.3864924139446682,
      "test_acc": 0.22962962962962963
    },
    {
      "epoch": 2,
      "train_loss": 1.404854349150986,
      "val_loss": 1.3866133860179

In [1]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import sys
import math
import json
import time
import random
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# FAST DEBUG CONFIG
# ============================================================

@dataclass
class FinetuneConfig:
    datasets_to_run: Tuple[str, ...] = ("anatomy",)

    cache_dir: str = "mcq_cache"
    out_dir: str = "runs/mcq_hlcm_mmlu_auxtrain"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 128

    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    epochs: int = 1
    train_batch_size: int = 16
    eval_batch_size: int = 16
    grad_accum_steps: int = 1

    head_lr: float = 3e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    max_grad_norm: float = 1.0
    label_smoothing: float = 0.10

    freeze_encoder: bool = True
    freeze_hlcm: bool = True
    head_dropout: float = 0.50

    seed: int = 42
    num_workers: int = 2
    use_bf16: bool = True
    prefer_gpu_index: int = 0

    aux_train_config_name: str = "auxiliary_train"
    aux_train_split_name: str = "train"

    subject_val_split_name: str = "validation"
    subject_test_split_name: str = "test"

    # FAST DEBUG LIMITS
    debug_train_limit: int = 1000
    debug_val_limit: int = 100
    debug_test_limit: int = 200


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds: float) -> str:
    seconds = max(float(seconds), 0.0)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def answerkey_to_index(answer_key: Any, num_choices: int) -> int:
    if isinstance(answer_key, int):
        idx = int(answer_key)
    else:
        answer_key = str(answer_key).strip()
        alpha = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
        numeric = {"0": 0, "1": 1, "2": 2, "3": 3, "4": 4}
        if answer_key in alpha:
            idx = alpha[answer_key]
        elif answer_key in numeric:
            idx = numeric[answer_key]
        else:
            raise ValueError(f"Unknown answer label: {answer_key}")

    if idx >= num_choices:
        raise ValueError(f"answer={answer_key} -> idx={idx}, but num_choices={num_choices}")
    return idx


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        print("[normalizer] not found")
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    print(f"[normalizer] loaded {normalizer_path}")
    return mu, sigma


def resolve_existing_split(ds_dict, preferred_name: str, fallback_names: List[str]) -> str:
    if preferred_name in ds_dict:
        return preferred_name
    for name in fallback_names:
        if name in ds_dict:
            return name
    raise KeyError(f"Could not find split '{preferred_name}'. Available: {list(ds_dict.keys())}")


# ============================================================
# LOSS
# ============================================================

def masked_choice_cross_entropy(
    logits: torch.Tensor,
    labels: torch.Tensor,
    choice_mask: torch.Tensor,
    label_smoothing: float = 0.0,
) -> torch.Tensor:
    b, _ = logits.shape
    log_probs = F.log_softmax(logits, dim=-1)

    losses = []
    for i in range(b):
        valid = choice_mask[i]
        valid_idx = torch.nonzero(valid, as_tuple=False).squeeze(-1)
        lp = log_probs[i, valid]

        gold_global = int(labels[i].item())
        local_pos = (valid_idx == gold_global).nonzero(as_tuple=False)

        if local_pos.numel() == 0:
            raise RuntimeError(f"Gold label {gold_global} not found among valid choices")

        local_pos = int(local_pos.item())
        cv = int(lp.size(0))

        if label_smoothing > 0:
            target = torch.full_like(lp, fill_value=label_smoothing / cv)
            target[local_pos] += 1.0 - label_smoothing
            loss_i = -(target * lp).sum()
        else:
            loss_i = -lp[local_pos]

        losses.append(loss_i)

    return torch.stack(losses).mean()


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(self, model_name, chunk_tok_len, seq_len, batch_size, device):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.encoder = AutoModel.from_pretrained(model_name).to(device)
        self.encoder.eval()

        for p in self.encoder.parameters():
            p.requires_grad = False

        self.embed_dim = int(self.encoder.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.embed_dim, dtype=torch.float32)

        inputs = self.tokenizer(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device, non_blocking=True) for k, v in inputs.items()}

        if self.device.type == "cuda":
            dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
            with amp.autocast(device_type="cuda", enabled=True, dtype=dtype):
                out = self.encoder(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.encoder(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _text_to_chunk_texts(self, text: str) -> List[str]:
        ids = self.tokenizer(text, add_special_tokens=False)["input_ids"]
        if len(ids) == 0:
            return []

        chunk_texts = []
        for i in range(0, len(ids), self.chunk_tok_len):
            chunk_ids = ids[i:i + self.chunk_tok_len]
            chunk_texts.append(
                self.tokenizer.decode(chunk_ids, clean_up_tokenization_spaces=True)
            )
        return chunk_texts

    def encode_text(self, text: str) -> torch.Tensor:
        chunk_texts = self._text_to_chunk_texts(text)

        if len(chunk_texts) == 0:
            return torch.zeros(self.seq_len, self.embed_dim, dtype=torch.float32)

        chunk_texts = chunk_texts[:self.seq_len]

        out = []
        for i in range(0, len(chunk_texts), self.batch_size):
            out.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        embs = torch.cat(out, dim=0)

        if embs.size(0) < self.seq_len:
            pad = torch.zeros(self.seq_len - embs.size(0), embs.size(1), dtype=embs.dtype)
            embs = torch.cat([embs, pad], dim=0)

        return embs[:self.seq_len]

    def encode_choice_set(self, question_stem: str, choice_texts: List[str]) -> torch.Tensor:
        feats = []
        for ch in choice_texts:
            text = f"Question: {question_stem}\nAnswer Choice: {ch}"
            feats.append(self.encode_text(text))
        return torch.stack(feats, dim=0)


# ============================================================
# DATA
# ============================================================

def load_mmlu_auxiliary_train():
    return load_dataset("cais/mmlu", "auxiliary_train")


def load_mmlu_subject(subject_name: str):
    return load_dataset("cais/mmlu", subject_name)


def unwrap_mmlu_row(example: Dict[str, Any]) -> Dict[str, Any]:
    if isinstance(example, dict) and len(example) == 1:
        only_key = next(iter(example.keys()))
        only_val = example[only_key]
        if only_key in {"train", "validation", "val", "test", "dev"} and isinstance(only_val, dict):
            return only_val
    return example


def normalize_choices_field(example: Dict[str, Any]) -> Tuple[str, List[str], int]:
    example = unwrap_mmlu_row(example)

    if "question" in example:
        stem = str(example["question"])
    elif "input" in example:
        stem = str(example["input"])
    elif "prompt" in example:
        stem = str(example["prompt"])
    else:
        raise KeyError(f"question field not found. keys={list(example.keys())}")

    if "choices" in example:
        raw_choices = example["choices"]
        if isinstance(raw_choices, list):
            choice_texts = [str(x) for x in raw_choices]
        elif isinstance(raw_choices, dict):
            if "text" in raw_choices:
                choice_texts = [str(x) for x in raw_choices["text"]]
            elif "label" in raw_choices:
                choice_texts = [str(x) for x in raw_choices["label"]]
            else:
                raise KeyError(f"unsupported choices dict: {raw_choices.keys()}")
        else:
            raise TypeError(f"Unsupported choices type: {type(raw_choices)}")
    elif all(k in example for k in ["A", "B", "C", "D"]):
        choice_texts = [str(example["A"]), str(example["B"]), str(example["C"]), str(example["D"])]
        if "E" in example:
            choice_texts.append(str(example["E"]))
    elif "options" in example:
        choice_texts = [str(x) for x in example["options"]]
    else:
        raise KeyError(f"choice field not found. keys={list(example.keys())}")

    if "answer" in example:
        answer_value = example["answer"]
    elif "answerKey" in example:
        answer_value = example["answerKey"]
    elif "target" in example:
        answer_value = example["target"]
    elif "label" in example and not isinstance(example["label"], list):
        answer_value = example["label"]
    else:
        raise KeyError(f"answer field not found. keys={list(example.keys())}")

    label = answerkey_to_index(answer_value, len(choice_texts))
    return stem, choice_texts, label


def build_or_load_cached_split(
    cache_name_prefix: str,
    split_name: str,
    split_data,
    conceptizer: DebertaConceptizer,
    cache_dir: str,
) -> List[Dict[str, Any]]:
    ensure_dir(cache_dir)
    safe_ds = cache_name_prefix.replace("/", "_")

    path = os.path.join(
        cache_dir,
        f"{safe_ds}_{split_name}_seq{conceptizer.seq_len}_tok{conceptizer.chunk_tok_len}.pt"
    )

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path, map_location="cpu")

    rows = []
    skipped = 0

    print(f"[cache] building {cache_name_prefix} / {split_name}")

    for ex in tqdm(split_data, desc=f"Conceptizing {cache_name_prefix}-{split_name}"):
        try:
            stem, choice_texts, label = normalize_choices_field(ex)
            x = conceptizer.encode_choice_set(stem, choice_texts)
            choice_mask = torch.ones(x.size(0), dtype=torch.bool)

            rows.append({
                "x": x,
                "label": int(label),
                "choice_mask": choice_mask,
                "num_choices": int(x.size(0)),
            })
        except Exception as e:
            skipped += 1
            print(f"[warn] skipped: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")
    return rows


class MCQFeatureDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "x": r["x"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "choice_mask": r["choice_mask"],
            "num_choices": r["num_choices"],
        }


def mcq_collate(batch):
    max_c = max(item["x"].size(0) for item in batch)
    t = batch[0]["x"].size(1)
    d = batch[0]["x"].size(2)
    b = len(batch)

    x = torch.zeros(b, max_c, t, d, dtype=torch.float32)
    choice_mask = torch.zeros(b, max_c, dtype=torch.bool)
    labels = torch.zeros(b, dtype=torch.long)

    for i, item in enumerate(batch):
        c = item["x"].size(0)
        x[i, :c] = item["x"]
        choice_mask[i, :c] = item["choice_mask"]
        labels[i] = item["label"]

    return {"x": x, "choice_mask": choice_mask, "labels": labels}


# ============================================================
# MODEL
# ============================================================

class MCQHead(nn.Module):
    def __init__(self, model: HyperbolicLCM, dropout: float = 0.5):
        super().__init__()
        self.model = model
        dim = model.layers[0].attn.embed_dim

        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dim, 1)

        nn.init.normal_(self.classifier.weight, mean=0.0, std=0.01)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, x, choice_mask, mu=None, sigma=None):
        b, c, t, din = x.shape
        x = x.reshape(b * c, t, din)

        if mu is not None and sigma is not None:
            x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

        with torch.no_grad():
            h = self.model(x)

        last_h = h[:, -1, :]
        last_tan = self.model.manifold.logmap0(last_h)
        last_tan = self.norm(last_tan)
        last_tan = self.dropout(last_tan)

        logits = self.classifier(last_tan).view(b, c)
        logits = logits.masked_fill(~choice_mask, -1e9)
        return logits


def build_hlcm_from_cfg(cfg: FinetuneConfig):
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: FinetuneConfig, device):
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"HLCM checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"[load] loaded pretrained HLCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    model.eval()
    for p in model.parameters():
        p.requires_grad = False

    return model


# ============================================================
# TRAIN / EVAL / INFERENCE
# ============================================================

@torch.no_grad()
def evaluate(model, loader, device, mu, sigma):
    model.eval()

    total_loss = 0.0
    total = 0
    correct = 0

    for batch in loader:
        x = batch["x"].to(device, non_blocking=True)
        choice_mask = batch["choice_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        logits = model(x, choice_mask, mu=mu, sigma=sigma)
        loss = masked_choice_cross_entropy(logits, labels, choice_mask, label_smoothing=0.0)

        preds = logits.argmax(dim=-1)

        total_loss += float(loss.item()) * x.size(0)
        correct += int((preds == labels).sum().item())
        total += int(x.size(0))

    return {
        "loss": total_loss / max(total, 1),
        "acc": correct / max(total, 1),
    }


@torch.no_grad()
def run_inference_with_time(model, loader, device, mu, sigma, save_path=None):
    model.eval()

    total = 0
    correct = 0
    predictions = []

    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    for batch in tqdm(loader, desc="Inference"):
        x = batch["x"].to(device, non_blocking=True)
        choice_mask = batch["choice_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        logits = model(x, choice_mask, mu=mu, sigma=sigma)
        preds = logits.argmax(dim=-1)

        batch_correct = preds.eq(labels)

        for i in range(x.size(0)):
            predictions.append({
                "example_index": total + i,
                "gold": int(labels[i].item()),
                "pred": int(preds[i].item()),
                "correct": int(batch_correct[i].item()),
            })

        correct += int(batch_correct.sum().item())
        total += int(x.size(0))

    if device.type == "cuda":
        torch.cuda.synchronize()

    inference_time_sec = time.perf_counter() - start_time

    results = {
        "num_examples": total,
        "correct": correct,
        "accuracy": correct / max(total, 1),
        "accuracy_percent": 100.0 * correct / max(total, 1),
        "inference_time_sec": inference_time_sec,
        "inference_time_hms": fmt_hms(inference_time_sec),
        "time_per_example_sec": inference_time_sec / max(total, 1),
        "examples_per_second": total / max(inference_time_sec, 1e-9),
        "predictions": predictions,
    }

    if save_path is not None:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=2)
        print(f"[save] inference results -> {save_path}")

    return results


# ============================================================
# MAIN FAST DEBUG TRAIN + INFERENCE
# ============================================================

def fast_debug_train_and_infer(cfg: FinetuneConfig):
    device = pick_device(cfg.prefer_gpu_index)
    set_seed(cfg.seed)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    subject_name = cfg.datasets_to_run[0]
    subject_safe = subject_name.replace("/", "_")
    out_dir = os.path.join(cfg.out_dir, subject_safe)

    ensure_dir(cfg.out_dir)
    ensure_dir(out_dir)
    ensure_dir(cfg.cache_dir)

    print(f"[device] {device}")
    print(f"[subject] {subject_name}")
    print(f"[out_dir] {out_dir}")

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=device,
    )

    aux_ds = load_mmlu_auxiliary_train()
    aux_train_split = resolve_existing_split(
        aux_ds,
        cfg.aux_train_split_name,
        ["train", "auxiliary_train"]
    )

    subject_ds = load_mmlu_subject(subject_name)
    val_split = resolve_existing_split(
        subject_ds,
        cfg.subject_val_split_name,
        ["validation", "val", "dev"]
    )
    test_split = resolve_existing_split(
        subject_ds,
        cfg.subject_test_split_name,
        ["test"]
    )

    train_rows = build_or_load_cached_split(
        f"mmlu_auxtrain_for_{subject_name}",
        aux_train_split,
        aux_ds[aux_train_split],
        conceptizer,
        cfg.cache_dir,
    )

    val_rows = build_or_load_cached_split(
        f"mmlu_{subject_name}",
        val_split,
        subject_ds[val_split],
        conceptizer,
        cfg.cache_dir,
    )

    test_rows = build_or_load_cached_split(
        f"mmlu_{subject_name}",
        test_split,
        subject_ds[test_split],
        conceptizer,
        cfg.cache_dir,
    )

    # FAST DEBUG SUBSET
    train_rows = train_rows[:cfg.debug_train_limit]
    val_rows = val_rows[:cfg.debug_val_limit]
    test_rows = test_rows[:cfg.debug_test_limit]

    print(f"[debug subset] train={len(train_rows)} val={len(val_rows)} test={len(test_rows)}")

    train_ds = MCQFeatureDataset(train_rows)
    val_ds = MCQFeatureDataset(val_rows)
    test_ds = MCQFeatureDataset(test_rows)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    hlcm = load_pretrained_hlcm(cfg, device)
    model = MCQHead(hlcm, dropout=cfg.head_dropout).to(device)

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        [{"params": trainable_params, "lr": cfg.head_lr, "weight_decay": cfg.weight_decay}]
    )

    best_val_acc = -1.0
    best_path = os.path.join(out_dir, "best_mcq.pt")

    use_amp = cfg.use_bf16 and device.type == "cuda"
    amp_dtype = torch.bfloat16

    start_train = time.time()

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        running_loss = 0.0
        seen = 0

        pbar = tqdm(train_loader, desc=f"Fast debug train epoch {epoch}/{cfg.epochs}")

        for batch in pbar:
            x = batch["x"].to(device, non_blocking=True)
            choice_mask = batch["choice_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=amp_dtype):
                    logits = model(x, choice_mask, mu=mu, sigma=sigma)
                    loss = masked_choice_cross_entropy(
                        logits,
                        labels,
                        choice_mask,
                        label_smoothing=cfg.label_smoothing,
                    )
            else:
                logits = model(x, choice_mask, mu=mu, sigma=sigma)
                loss = masked_choice_cross_entropy(
                    logits,
                    labels,
                    choice_mask,
                    label_smoothing=cfg.label_smoothing,
                )

            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True)
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

            running_loss += float(loss.item()) * x.size(0)
            seen += int(x.size(0))

            pbar.set_postfix(loss=f"{running_loss / max(seen, 1):.4f}")

        train_loss = running_loss / max(seen, 1)
        val_metrics = evaluate(model, val_loader, device, mu, sigma)

        print(
            f"[epoch {epoch}] "
            f"train_loss={train_loss:.4f} "
            f"val_loss={val_metrics['loss']:.4f} "
            f"val_acc={val_metrics['acc']:.4f}"
        )

        if val_metrics["acc"] > best_val_acc:
            best_val_acc = float(val_metrics["acc"])

            torch.save(
                {
                    "cfg": asdict(cfg),
                    "subject_name": subject_name,
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "hlcm_state": model.model.state_dict(),
                    "best_val_acc": best_val_acc,
                },
                best_path,
            )

            print(f"[save] best checkpoint -> {best_path}")

    train_time = time.time() - start_train

    # Reload best checkpoint before inference
    best_obj = torch.load(best_path, map_location=device)
    model.load_state_dict(best_obj["model_state"], strict=True)
    model.eval()

    inference_save_path = os.path.join(out_dir, "fast_debug_inference_results.json")

    inference_results = run_inference_with_time(
        model=model,
        loader=test_loader,
        device=device,
        mu=mu,
        sigma=sigma,
        save_path=inference_save_path,
    )

    summary = {
        "subject_name": subject_name,
        "best_checkpoint": best_path,
        "train_time_sec": train_time,
        "train_time_hms": fmt_hms(train_time),
        "debug_train_examples": len(train_rows),
        "debug_val_examples": len(val_rows),
        "debug_test_examples": len(test_rows),
        "best_val_acc": best_val_acc,
        "test_accuracy_percent": inference_results["accuracy_percent"],
        "inference_time_sec": inference_results["inference_time_sec"],
        "inference_time_hms": inference_results["inference_time_hms"],
        "time_per_example_sec": inference_results["time_per_example_sec"],
        "examples_per_second": inference_results["examples_per_second"],
    }

    summary_path = os.path.join(out_dir, "fast_debug_summary.json")
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)

    print("\n==================== FAST DEBUG DONE ====================")
    print(json.dumps(summary, indent=2))
    print(f"[summary saved] {summary_path}")

    return summary, inference_results

In [2]:
cfg = FinetuneConfig(
    datasets_to_run=("anatomy",),

    # keep this path same as your pretrained HLCM checkpoint
    ckpt_path="runs/hyperbolic_cluster/checkpoints/ckpt_best.pt",

    normalizer_path="normalizer.pt",
    out_dir="runs/mcq_hlcm_mmlu_auxtrain",
    cache_dir="mcq_cache",

    epochs=1,

    # Faster debug settings
    train_batch_size=16,
    eval_batch_size=16,
    encoder_batch_size=128,

    # Reduce these more if it is still slow
    debug_train_limit=1000,
    debug_val_limit=100,
    debug_test_limit=200,

    prefer_gpu_index=0,
    seed=42,
)

summary, inference_results = fast_debug_train_and_infer(cfg)

[device] cuda:0
[subject] anatomy
[out_dir] runs/mcq_hlcm_mmlu_auxtrain/anatomy


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Using the latest cached version of the dataset since

[cache] loading mcq_cache/mmlu_auxtrain_for_anatomy_train_seq8_tok256.pt
[cache] loading mcq_cache/mmlu_anatomy_validation_seq8_tok256.pt
[cache] loading mcq_cache/mmlu_anatomy_test_seq8_tok256.pt
[debug subset] train=1000 val=14 test=135
[load] loaded pretrained HLCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] missing keys: 0
[load] unexpected keys: 0
[normalizer] loaded normalizer.pt


Fast debug train epoch 1/1: 100%|██████████████████████| 63/63 [00:17<00:00,  3.62it/s, loss=1.6420]


[epoch 1] train_loss=1.6420 val_loss=1.3850 val_acc=0.2857
[save] best checkpoint -> runs/mcq_hlcm_mmlu_auxtrain/anatomy/best_mcq.pt


Inference: 100%|██████████████████████████████████████████████████████| 9/9 [00:02<00:00,  3.99it/s]


[save] inference results -> runs/mcq_hlcm_mmlu_auxtrain/anatomy/fast_debug_inference_results.json

==================== FAST DEBUG DONE ====================
{
  "subject_name": "anatomy",
  "best_checkpoint": "runs/mcq_hlcm_mmlu_auxtrain/anatomy/best_mcq.pt",
  "train_time_sec": 30.272852897644043,
  "train_time_hms": "00:00:30",
  "debug_train_examples": 1000,
  "debug_val_examples": 14,
  "debug_test_examples": 135,
  "best_val_acc": 0.2857142857142857,
  "test_accuracy_percent": 28.14814814814815,
  "inference_time_sec": 2.2564579052850604,
  "inference_time_hms": "00:00:02",
  "time_per_example_sec": 0.01671450300211156,
  "examples_per_second": 59.82828205383487
}
[summary saved] runs/mcq_hlcm_mmlu_auxtrain/anatomy/fast_debug_summary.json


In [3]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import sys
import math
import json
import time
import random
import argparse
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class FinetuneConfig:

    datasets_to_run: Tuple[str, ...] = ("commonsense_qa",)

    cache_dir: str = "mcq_cache"

    encoder_name: str = "microsoft/deberta-v3-small"

    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 64

    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4

    dropout: float = 0.30
    manifold_c: float = 0.002

    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    epochs: int = 10

    train_batch_size: int = 4
    eval_batch_size: int = 8

    grad_accum_steps: int = 1

    lr: float = 0.0
    head_lr: float = 3e-5

    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    max_grad_norm: float = 1.0

    label_smoothing: float = 0.10

    freeze_encoder: bool = True
    freeze_hlcm: bool = True

    head_dropout: float = 0.50

    seed: int = 42
    num_workers: int = 4

    use_bf16: bool = True
    prefer_gpu_index: int = 0

    out_dir: str = "runs/mcq_hlcm_commonsenseqa"


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):

    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:

    if not torch.cuda.is_available():
        return torch.device("cpu")

    n = torch.cuda.device_count()

    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))

    torch.cuda.set_device(idx)

    return torch.device(f"cuda:{idx}")


def ensure_dir(path: str):

    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds: float) -> str:

    seconds = max(float(seconds), 0.0)

    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)

    return f"{h:02d}:{m:02d}:{s:02d}"


def answerkey_to_index(answer_key: str, num_choices: int) -> int:

    answer_key = str(answer_key).strip()

    alpha = {"A":0,"B":1,"C":2,"D":3,"E":4}

    idx = alpha[answer_key]

    if idx >= num_choices:
        raise ValueError()

    return idx


def load_normalizer(normalizer_path: str, device: torch.device):

    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")

    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)

    return mu, sigma


# ============================================================
# MASKED LOSS
# ============================================================

def masked_choice_cross_entropy(logits, labels, choice_mask, label_smoothing=0.0):

    log_probs = F.log_softmax(logits, dim=-1)

    losses = []

    for i in range(logits.size(0)):

        valid = choice_mask[i]

        valid_idx = torch.nonzero(valid).squeeze(-1)

        lp = log_probs[i, valid]

        gold_global = labels[i].item()

        local_pos = (valid_idx == gold_global).nonzero().item()

        if label_smoothing > 0:

            target = torch.full_like(lp, label_smoothing / lp.size(0))

            target[local_pos] += (1.0 - label_smoothing)

            loss = -(target * lp).sum()

        else:

            loss = -lp[local_pos]

        losses.append(loss)

    return torch.stack(losses).mean()


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:

    def __init__(self, model_name, chunk_tok_len, seq_len, batch_size, device):

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        self.encoder = AutoModel.from_pretrained(model_name).to(device)

        self.encoder.eval()

        for p in self.encoder.parameters():
            p.requires_grad=False

        self.chunk_tok_len = chunk_tok_len
        self.seq_len = seq_len
        self.batch_size = batch_size
        self.device = device

        self.embed_dim = self.encoder.config.hidden_size


    @torch.inference_mode()
    def embed(self, texts):

        inputs = self.tokenizer(texts,padding=True,truncation=True,max_length=self.chunk_tok_len,return_tensors="pt")

        inputs = {k:v.to(self.device) for k,v in inputs.items()}

        with amp.autocast(device_type="cuda",enabled=(self.device.type=="cuda"),dtype=torch.bfloat16):

            out = self.encoder(**inputs).last_hidden_state[:,0,:]

        return out.detach().cpu()


    def encode_text(self,text):

        ids = self.tokenizer(text,add_special_tokens=False)["input_ids"]

        chunks=[ids[i:i+self.chunk_tok_len] for i in range(0,len(ids),self.chunk_tok_len)]

        texts=[self.tokenizer.decode(c) for c in chunks[:self.seq_len]]

        if len(texts)==0:
            return torch.zeros(self.seq_len,self.embed_dim)

        embs=[]

        for i in range(0,len(texts),self.batch_size):

            embs.append(self.embed(texts[i:i+self.batch_size]))

        embs=torch.cat(embs)

        if embs.size(0)<self.seq_len:

            pad=torch.zeros(self.seq_len-embs.size(0),self.embed_dim)

            embs=torch.cat([embs,pad])

        return embs[:self.seq_len]


    def encode_choice_set(self,question,choices):

        feats=[]

        for c in choices:

            text=f"Question: {question}\nAnswer Choice: {c}"

            feats.append(self.encode_text(text))

        return torch.stack(feats)


# ============================================================
# DATA PREP
# ============================================================

def load_raw_mcq_dataset(name):

    if name=="commonsense_qa":

        return load_dataset("commonsense_qa")

    raise ValueError()


def normalize_choices_field(example):

    stem=str(example["question"])

    choice_texts=list(example["choices"]["text"])

    label=answerkey_to_index(example["answerKey"],len(choice_texts))

    return stem,choice_texts,label


# ============================================================
# Dataset wrapper
# ============================================================

class MCQFeatureDataset(Dataset):

    def __init__(self,rows):
        self.rows=rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self,idx):

        r=self.rows[idx]

        return {
            "x":r["x"],
            "label":torch.tensor(r["label"]),
            "choice_mask":r["choice_mask"]
        }


def mcq_collate(batch):

    max_c=max(item["x"].size(0) for item in batch)

    b=len(batch)
    t=batch[0]["x"].size(1)
    d=batch[0]["x"].size(2)

    x=torch.zeros(b,max_c,t,d)

    mask=torch.zeros(b,max_c,dtype=torch.bool)

    labels=torch.zeros(b,dtype=torch.long)

    for i,item in enumerate(batch):

        c=item["x"].size(0)

        x[i,:c]=item["x"]

        mask[i,:c]=True

        labels[i]=item["label"]

    return {"x":x,"choice_mask":mask,"labels":labels}


# ============================================================
# MODEL
# ============================================================

class MCQHead(nn.Module):

    def __init__(self,model,dropout):

        super().__init__()

        self.model=model

        dim=model.layers[0].attn.embed_dim

        self.norm=nn.LayerNorm(dim)

        self.dropout=nn.Dropout(dropout)

        self.classifier=nn.Linear(dim,1)


    def forward(self,x,mask,mu=None,sigma=None):

        b,c,t,d=x.shape

        x=x.reshape(b*c,t,d)

        with torch.no_grad():

            h=self.model(x)

        h=h[:,-1]

        h=self.model.manifold.logmap0(h)

        h=self.norm(h)

        h=self.dropout(h)

        logits=self.classifier(h).view(b,c)

        logits=logits.masked_fill(~mask,-1e9)

        return logits


# ============================================================
# HLCM Loader
# ============================================================

def build_hlcm(cfg):

    return HyperbolicLCM(

        in_dim=cfg.in_dim,

        model_dim=cfg.model_dim,

        num_heads=cfg.num_heads,

        num_layers=cfg.num_layers,

        ffn_mult=cfg.ffn_mult,

        dropout=cfg.dropout,

        manifold_c=cfg.manifold_c,

        causal=cfg.causal,

        input_scale=cfg.input_scale,

        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg,device):

    model=build_hlcm(cfg).to(device)

    ckpt=torch.load(cfg.ckpt_path,map_location="cpu")

    state=ckpt["model"] if "model" in ckpt else ckpt

    model.load_state_dict(state,strict=False)

    model.eval()

    for p in model.parameters():
        p.requires_grad=False

    return model


# ============================================================
# TRAIN / EVAL
# ============================================================

@torch.no_grad()
def evaluate(model,loader,device,mu,sigma):

    model.eval()

    total=0
    correct=0
    total_loss=0

    for batch in loader:

        x=batch["x"].to(device)
        mask=batch["choice_mask"].to(device)
        labels=batch["labels"].to(device)

        logits=model(x,mask,mu,sigma)

        loss=masked_choice_cross_entropy(logits,labels,mask)

        total_loss+=loss.item()*x.size(0)

        preds=logits.argmax(-1)

        correct+=(preds==labels).sum().item()

        total+=x.size(0)

    model.train()

    return {"loss":total_loss/total,"acc":correct/total}


# ============================================================
# MAIN TRAIN FUNCTION
# ============================================================

def train_one_dataset(dataset_name,cfg):

    device=pick_device(cfg.prefer_gpu_index)

    set_seed(cfg.seed)

    conceptizer=DebertaConceptizer(
        cfg.encoder_name,
        cfg.chunk_tok_len,
        cfg.seq_len,
        cfg.encoder_batch_size,
        device
    )

    raw=load_raw_mcq_dataset(dataset_name)

    def build(split):

        rows=[]

        for ex in tqdm(raw[split]):

            stem,choices,label=normalize_choices_field(ex)

            x=conceptizer.encode_choice_set(stem,choices)

            rows.append({
                "x":x,
                "label":label,
                "choice_mask":torch.ones(x.size(0),dtype=torch.bool)
            })

        return rows


    train_rows=build("train")
    val_rows=build("validation")

    train_ds=MCQFeatureDataset(train_rows)
    val_ds=MCQFeatureDataset(val_rows)

    train_loader=DataLoader(train_ds,batch_size=cfg.train_batch_size,shuffle=True,collate_fn=mcq_collate)
    val_loader=DataLoader(val_ds,batch_size=cfg.eval_batch_size,collate_fn=mcq_collate)

    hlcm=load_pretrained_hlcm(cfg,device)

    model=MCQHead(hlcm,cfg.head_dropout).to(device)

    mu,sigma=load_normalizer(cfg.normalizer_path,device)

    optimizer=torch.optim.AdamW(model.parameters(),lr=cfg.head_lr)

    for epoch in range(cfg.epochs):

        model.train()

        for batch in tqdm(train_loader):

            x=batch["x"].to(device)
            mask=batch["choice_mask"].to(device)
            labels=batch["labels"].to(device)

            logits=model(x,mask,mu,sigma)

            loss=masked_choice_cross_entropy(
                logits,
                labels,
                mask,
                cfg.label_smoothing
            )

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

        val=evaluate(model,val_loader,device,mu,sigma)

        print(f"Epoch {epoch+1} val_acc={val['acc']:.4f}")


# ============================================================
# MAIN
# ============================================================

def main():

    cfg=FinetuneConfig()

    for ds in cfg.datasets_to_run:

        train_one_dataset(ds,cfg)


if __name__=="__main__":
    main()

/home/user/anaconda3/envs/raat/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
100%|████████████████████████████████████████████████████████████████████████████████████████| 2436/2436 [03:54<00:00, 10.37it/s]


Epoch 1 val_acc=0.1245


100%|████████████████████████████████████████████████████████████████████████████████████████| 2436/2436 [03:45<00:00, 10.82it/s]


Epoch 2 val_acc=0.1597


100%|████████████████████████████████████████████████████████████████████████████████████████| 2436/2436 [03:44<00:00, 10.85it/s]


...

Epoch 8 val_acc=0.2891


100%|████████████████████████████████████████████████████████████████████████████████████████| 2436/2436 [03:43<00:00, 10.89it/s]


Epoch 9 val_acc=0.2924


100%|████████████████████████████████████████████████████████████████████████████████████████| 2436/2436 [03:44<00:00, 10.87it/s]


Epoch 10 val_acc=0.2801


In [1]:
#inference
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import json
import time
import random
from dataclasses import dataclass
from typing import List, Dict, Any, Optional, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class InferenceConfig:
    dataset_name: str = "commonsense_qa"

    cache_dir: str = "mcq_cache"
    out_dir: str = "runs/mcq_hlcm_commonsenseqa"

    # Fine-tuned checkpoint. If missing, code still runs for timing with random MCQ head.
    best_ckpt_path: str = "runs/mcq_hlcm_commonsenseqa/commonsense_qa/best_mcq.pt"

    # Pretrained HLCM backbone checkpoint.
    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 64

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    head_dropout: float = 0.50

    eval_batch_size: int = 8
    num_workers: int = 4
    prefer_gpu_index: int = 0
    seed: int = 42

    # CommonSenseQA test labels are usually unavailable, so validation is best for accuracy.
    split: str = "validation"

    build_cache_if_missing: bool = True


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")

    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds: float) -> str:
    seconds = max(float(seconds), 0.0)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def answerkey_to_index(answer_key: str, num_choices: int) -> int:
    answer_key = str(answer_key).strip()
    alpha = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

    if answer_key not in alpha:
        raise ValueError(f"Unknown answerKey: {answer_key}")

    idx = alpha[answer_key]

    if idx >= num_choices:
        raise ValueError(f"answerKey={answer_key} gives idx={idx}, but num_choices={num_choices}")

    return idx


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        print("[normalizer] not found; using None")
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)

    print(f"[normalizer] loaded from {normalizer_path}")
    return mu, sigma


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(self, model_name, chunk_tok_len, seq_len, batch_size, device):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.encoder = AutoModel.from_pretrained(model_name).to(device)
        self.encoder.eval()

        for p in self.encoder.parameters():
            p.requires_grad = False

        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device
        self.embed_dim = int(self.encoder.config.hidden_size)

    @torch.inference_mode()
    def embed(self, texts: List[str]) -> torch.Tensor:
        if len(texts) == 0:
            return torch.empty(0, self.embed_dim, dtype=torch.float32)

        inputs = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )

        inputs = {k: v.to(self.device, non_blocking=True) for k, v in inputs.items()}

        if self.device.type == "cuda":
            dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
            with amp.autocast(device_type="cuda", enabled=True, dtype=dtype):
                out = self.encoder(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.encoder(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def encode_text(self, text: str) -> torch.Tensor:
        ids = self.tokenizer(text, add_special_tokens=False)["input_ids"]

        if len(ids) == 0:
            return torch.zeros(self.seq_len, self.embed_dim, dtype=torch.float32)

        chunks = [ids[i:i + self.chunk_tok_len] for i in range(0, len(ids), self.chunk_tok_len)]
        texts = [
            self.tokenizer.decode(c, clean_up_tokenization_spaces=True)
            for c in chunks[:self.seq_len]
        ]

        embs = []
        for i in range(0, len(texts), self.batch_size):
            embs.append(self.embed(texts[i:i + self.batch_size]))

        embs = torch.cat(embs, dim=0)

        if embs.size(0) < self.seq_len:
            pad = torch.zeros(self.seq_len - embs.size(0), self.embed_dim, dtype=embs.dtype)
            embs = torch.cat([embs, pad], dim=0)

        return embs[:self.seq_len]

    def encode_choice_set(self, question: str, choices: List[str]) -> torch.Tensor:
        feats = []
        for c in choices:
            text = f"Question: {question}\nAnswer Choice: {c}"
            feats.append(self.encode_text(text))
        return torch.stack(feats, dim=0)


# ============================================================
# DATA
# ============================================================

def load_raw_mcq_dataset(name: str):
    if name == "commonsense_qa":
        return load_dataset("commonsense_qa")
    raise ValueError(f"Unknown dataset: {name}")


def normalize_choices_field(example: Dict[str, Any]) -> Tuple[str, List[str], Optional[int]]:
    stem = str(example["question"])
    choice_texts = list(example["choices"]["text"])

    # Test split may not contain labels.
    if "answerKey" not in example or example["answerKey"] is None or str(example["answerKey"]).strip() == "":
        label = None
    else:
        label = answerkey_to_index(example["answerKey"], len(choice_texts))

    return stem, choice_texts, label


def cache_path_for_split(cfg: InferenceConfig, split_name: str) -> str:
    safe_ds = cfg.dataset_name.replace("/", "_")
    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_seq{cfg.seq_len}_tok{cfg.chunk_tok_len}.pt",
    )


def build_or_load_cached_split(
    cfg: InferenceConfig,
    split_name: str,
    split_data,
    conceptizer: Optional[DebertaConceptizer],
) -> List[Dict[str, Any]]:
    ensure_dir(cfg.cache_dir)
    path = cache_path_for_split(cfg, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path, map_location="cpu")

    if not cfg.build_cache_if_missing:
        raise FileNotFoundError(f"Cache not found: {path}")

    if conceptizer is None:
        raise RuntimeError("Conceptizer is required to build cache.")

    rows = []
    skipped = 0

    print(f"[cache] building {cfg.dataset_name} / {split_name}")

    for ex in tqdm(split_data, desc=f"Conceptizing {cfg.dataset_name}-{split_name}"):
        try:
            stem, choices, label = normalize_choices_field(ex)
            x = conceptizer.encode_choice_set(stem, choices)
            choice_mask = torch.ones(x.size(0), dtype=torch.bool)

            rows.append({
                "x": x,
                "label": -1 if label is None else int(label),
                "has_label": label is not None,
                "choice_mask": choice_mask,
                "num_choices": int(x.size(0)),
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")

    return rows


class MCQFeatureDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "x": r["x"],
            "label": torch.tensor(r.get("label", -1), dtype=torch.long),
            "has_label": torch.tensor(bool(r.get("has_label", r.get("label", -1) != -1)), dtype=torch.bool),
            "choice_mask": r["choice_mask"],
        }


def mcq_collate(batch):
    max_c = max(item["x"].size(0) for item in batch)
    b = len(batch)
    t = batch[0]["x"].size(1)
    d = batch[0]["x"].size(2)

    x = torch.zeros(b, max_c, t, d, dtype=torch.float32)
    mask = torch.zeros(b, max_c, dtype=torch.bool)
    labels = torch.full((b,), -1, dtype=torch.long)
    has_label = torch.zeros(b, dtype=torch.bool)

    for i, item in enumerate(batch):
        c = item["x"].size(0)
        x[i, :c] = item["x"]
        mask[i, :c] = item["choice_mask"]
        labels[i] = item["label"]
        has_label[i] = item["has_label"]

    return {
        "x": x,
        "choice_mask": mask,
        "labels": labels,
        "has_label": has_label,
    }


# ============================================================
# MODEL
# ============================================================

class MCQHead(nn.Module):
    def __init__(self, model, dropout):
        super().__init__()

        self.model = model
        dim = model.layers[0].attn.embed_dim

        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dim, 1)

    def forward(self, x, mask, mu=None, sigma=None):
        b, c, t, d = x.shape
        x = x.reshape(b * c, t, d)

        if mu is not None and sigma is not None:
            x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

        with torch.no_grad():
            h = self.model(x)

        h = h[:, -1]
        h = self.model.manifold.logmap0(h)
        h = self.norm(h)
        h = self.dropout(h)

        logits = self.classifier(h).view(b, c)
        logits = logits.masked_fill(~mask, -1e9)

        return logits


def build_hlcm(cfg):
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_inference_model(cfg: InferenceConfig, device: torch.device):
    hlcm = build_hlcm(cfg).to(device)
    model = MCQHead(hlcm, cfg.head_dropout).to(device)

    if os.path.exists(cfg.best_ckpt_path):
        print(f"[load] loading fine-tuned checkpoint: {cfg.best_ckpt_path}")
        obj = torch.load(cfg.best_ckpt_path, map_location=device)

        if "model_state" not in obj:
            raise KeyError("Checkpoint exists, but does not contain 'model_state'.")

        missing, unexpected = model.load_state_dict(obj["model_state"], strict=False)

        print(f"[load] fine-tuned model loaded")
        print(f"[load] missing keys: {len(missing)}")
        print(f"[load] unexpected keys: {len(unexpected)}")

    else:
        print("=" * 80)
        print("[warning] fine-tuned best_mcq.pt not found.")
        print("[warning] using pretrained HLCM backbone + random MCQ head.")
        print("[warning] inference time is valid, but accuracy is NOT meaningful.")
        print("=" * 80)

        if os.path.exists(cfg.ckpt_path):
            ckpt = torch.load(cfg.ckpt_path, map_location="cpu")
            state = ckpt["model"] if "model" in ckpt else ckpt
            missing, unexpected = hlcm.load_state_dict(state, strict=False)

            print(f"[load] pretrained HLCM loaded from {cfg.ckpt_path}")
            print(f"[load] HLCM missing keys: {len(missing)}")
            print(f"[load] HLCM unexpected keys: {len(unexpected)}")
        else:
            print("[warning] pretrained HLCM checkpoint also not found.")
            print("[warning] using fully random HLCM + random MCQ head.")

    model.eval()

    for p in model.parameters():
        p.requires_grad = False

    return model


# ============================================================
# INFERENCE
# ============================================================

@torch.no_grad()
def run_inference_with_time(model, loader, device, mu=None, sigma=None, save_path=None):
    model.eval()

    total = 0
    labeled_total = 0
    correct = 0
    predictions = []

    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    for batch in tqdm(loader, desc="Inference"):
        x = batch["x"].to(device, non_blocking=True)
        mask = batch["choice_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        has_label = batch["has_label"].to(device, non_blocking=True)

        logits = model(x, mask, mu=mu, sigma=sigma)
        preds = logits.argmax(dim=-1)

        if has_label.any():
            labeled_total += int(has_label.sum().item())
            correct += int((preds[has_label] == labels[has_label]).sum().item())

        for i in range(x.size(0)):
            item = {
                "example_index": total + i,
                "pred": int(preds[i].item()),
                "has_label": bool(has_label[i].item()),
            }

            if bool(has_label[i].item()):
                item["gold"] = int(labels[i].item())
                item["correct"] = int(preds[i].item() == labels[i].item())

            predictions.append(item)

        total += int(x.size(0))

    if device.type == "cuda":
        torch.cuda.synchronize()

    inference_time_sec = time.perf_counter() - start_time

    results = {
        "num_examples": total,
        "num_labeled_examples": labeled_total,
        "correct": correct if labeled_total > 0 else None,
        "accuracy": None if labeled_total == 0 else correct / labeled_total,
        "accuracy_percent": None if labeled_total == 0 else 100.0 * correct / labeled_total,
        "inference_time_sec": inference_time_sec,
        "inference_time_hms": fmt_hms(inference_time_sec),
        "time_per_example_sec": inference_time_sec / max(total, 1),
        "examples_per_second": total / max(inference_time_sec, 1e-9),
        "predictions": predictions,
    }

    if save_path is not None:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=2)

        print(f"[save] inference results -> {save_path}")

    return results


# ============================================================
# MAIN NOTEBOOK FUNCTION
# ============================================================

def inference_only_commonsenseqa(cfg: InferenceConfig):
    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print(f"[device] {device}")

    ensure_dir(cfg.cache_dir)

    dataset_out_dir = os.path.join(cfg.out_dir, cfg.dataset_name.replace("/", "_"))
    ensure_dir(dataset_out_dir)

    raw = load_raw_mcq_dataset(cfg.dataset_name)

    cache_path = cache_path_for_split(cfg, cfg.split)
    cache_build_time_sec = 0.0

    if not os.path.exists(cache_path):
        print(f"[cache] not found: {cache_path}")

        if not cfg.build_cache_if_missing:
            raise FileNotFoundError(cache_path)

        conceptizer = DebertaConceptizer(
            cfg.encoder_name,
            cfg.chunk_tok_len,
            cfg.seq_len,
            cfg.encoder_batch_size,
            device,
        )

        if device.type == "cuda":
            torch.cuda.synchronize()

        cache_start = time.perf_counter()

        rows = build_or_load_cached_split(
            cfg=cfg,
            split_name=cfg.split,
            split_data=raw[cfg.split],
            conceptizer=conceptizer,
        )

        if device.type == "cuda":
            torch.cuda.synchronize()

        cache_build_time_sec = time.perf_counter() - cache_start

    else:
        rows = build_or_load_cached_split(
            cfg=cfg,
            split_name=cfg.split,
            split_data=raw[cfg.split],
            conceptizer=None,
        )

    print(f"[data] split={cfg.split}, examples={len(rows)}")

    ds = MCQFeatureDataset(rows)

    loader = DataLoader(
        ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    model = load_inference_model(cfg, device)
    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    save_path = os.path.join(
        dataset_out_dir,
        f"inference_only_{cfg.split}_results.json",
    )

    inference_results = run_inference_with_time(
        model=model,
        loader=loader,
        device=device,
        mu=mu,
        sigma=sigma,
        save_path=save_path,
    )

    summary = {
        "dataset_name": cfg.dataset_name,
        "split": cfg.split,
        "checkpoint": cfg.best_ckpt_path,
        "cache_file": cache_path,
        "num_examples": inference_results["num_examples"],
        "num_labeled_examples": inference_results["num_labeled_examples"],
        "correct": inference_results["correct"],
        "accuracy": inference_results["accuracy"],
        "accuracy_percent": inference_results["accuracy_percent"],
        "cache_build_time_sec": cache_build_time_sec,
        "cache_build_time_hms": fmt_hms(cache_build_time_sec),
        "inference_time_sec": inference_results["inference_time_sec"],
        "inference_time_hms": inference_results["inference_time_hms"],
        "time_per_example_sec": inference_results["time_per_example_sec"],
        "examples_per_second": inference_results["examples_per_second"],
    }

    summary_path = os.path.join(
        dataset_out_dir,
        f"inference_only_{cfg.split}_summary.json",
    )

    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)

    print("\n==================== INFERENCE DONE ====================")
    print(json.dumps(summary, indent=2))
    print(f"[summary saved] {summary_path}")

    return summary, inference_results

In [2]:
cfg = InferenceConfig(
    dataset_name="commonsense_qa",

    best_ckpt_path="runs/mcq_hlcm_commonsenseqa/commonsense_qa/best_mcq.pt",
    ckpt_path="runs/hyperbolic_cluster/checkpoints/ckpt_best.pt",
    normalizer_path="normalizer.pt",

    cache_dir="mcq_cache",
    out_dir="runs/mcq_hlcm_commonsenseqa",

    split="validation",
    eval_batch_size=8,
    prefer_gpu_index=0,

    build_cache_if_missing=True,
)

summary, inference_results = inference_only_commonsenseqa(cfg)

[device] cuda:0


Using the latest cached version of the dataset since commonsense_qa couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /home/user/.cache/huggingface/datasets/commonsense_qa/default/0.0.0/94630fe30dad47192a8546eb75f094926d47e155 (last modified on Mon Mar  9 15:29:08 2026).


[cache] not found: mcq_cache/commonsense_qa_validation_seq8_tok256.pt


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[cache] building commonsense_qa / validation


Conceptizing commonsense_qa-validation: 100%|███████████████████| 1221/1221 [00:30<00:00, 40.30it/s]


[cache] saved mcq_cache/commonsense_qa_validation_seq8_tok256.pt (1221 examples, skipped=0)
[data] split=validation, examples=1221
[warning] fine-tuned best_mcq.pt not found.
[warning] using pretrained HLCM backbone + random MCQ head.
[warning] inference time is valid, but accuracy is NOT meaningful.
[load] pretrained HLCM loaded from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] HLCM missing keys: 0
[load] HLCM unexpected keys: 0
[normalizer] loaded from normalizer.pt


Inference: 100%|██████████████████████████████████████████████████| 153/153 [00:18<00:00,  8.31it/s]


[save] inference results -> runs/mcq_hlcm_commonsenseqa/commonsense_qa/inference_only_validation_results.json

==================== INFERENCE DONE ====================
{
  "dataset_name": "commonsense_qa",
  "split": "validation",
  "checkpoint": "runs/mcq_hlcm_commonsenseqa/commonsense_qa/best_mcq.pt",
  "cache_file": "mcq_cache/commonsense_qa_validation_seq8_tok256.pt",
  "num_examples": 1221,
  "num_labeled_examples": 1221,
  "correct": 246,
  "accuracy": 0.20147420147420148,
  "accuracy_percent": 20.147420147420146,
  "cache_build_time_sec": 30.442058730870485,
  "cache_build_time_hms": "00:00:30",
  "inference_time_sec": 18.485472416039556,
  "inference_time_hms": "00:00:18",
  "time_per_example_sec": 0.015139617048353446,
  "examples_per_second": 66.05186886868833
}
[summary saved] runs/mcq_hlcm_commonsenseqa/commonsense_qa/inference_only_validation_summary.json


In [3]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import sys
import math
import json
import time
import random
import argparse
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class FinetuneConfig:
    datasets_to_run: Tuple[str, ...] = ("ARC-Easy", "ARC-Challenge")
    cache_dir: str = "mcq_cache"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 64

    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    epochs: int = 3
    train_batch_size: int = 4
    eval_batch_size: int = 8
    grad_accum_steps: int = 1
    lr: float = 0.0
    head_lr: float = 3e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    max_grad_norm: float = 1.0
    label_smoothing: float = 0.10

    freeze_encoder: bool = True
    freeze_hlcm: bool = True

    head_dropout: float = 0.50

    seed: int = 42
    num_workers: int = 4
    use_bf16: bool = True
    prefer_gpu_index: int = 0

    out_dir: str = "runs/mcq_hlcm_arc_only"


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds: float) -> str:
    seconds = max(float(seconds), 0.0)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def answerkey_to_index(answer_key: str, num_choices: int) -> int:
    answer_key = str(answer_key).strip()
    alpha = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
    numeric = {"1": 0, "2": 1, "3": 2, "4": 3, "5": 4}

    if answer_key in alpha:
        idx = alpha[answer_key]
    elif answer_key in numeric:
        idx = numeric[answer_key]
    else:
        raise ValueError(f"Unknown answerKey: {answer_key}")

    if idx >= num_choices:
        raise ValueError(f"answerKey={answer_key} -> idx={idx}, but num_choices={num_choices}")
    return idx


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None
    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


# ============================================================
# MASKED LOSS
# ============================================================

def masked_choice_cross_entropy(
    logits: torch.Tensor,         # [B, C]
    labels: torch.Tensor,         # [B]
    choice_mask: torch.Tensor,    # [B, C] bool
    label_smoothing: float = 0.0,
) -> torch.Tensor:
    """
    Cross-entropy over only valid choices.
    Label smoothing mass is distributed only over valid choices,
    never onto padded choices.
    """
    b, _ = logits.shape
    log_probs = F.log_softmax(logits, dim=-1)

    losses = []
    for i in range(b):
        valid = choice_mask[i]
        valid_idx = torch.nonzero(valid, as_tuple=False).squeeze(-1)
        lp = log_probs[i, valid]  # [Cv]

        gold_global = int(labels[i].item())
        local_pos = (valid_idx == gold_global).nonzero(as_tuple=False)
        if local_pos.numel() == 0:
            raise RuntimeError(f"Gold label {gold_global} not found among valid choices in row {i}")
        local_pos = int(local_pos.item())

        cv = int(lp.size(0))
        if label_smoothing > 0.0:
            target = torch.full_like(lp, fill_value=label_smoothing / cv)
            target[local_pos] += (1.0 - label_smoothing)
            loss_i = -(target * lp).sum()
        else:
            loss_i = -lp[local_pos]

        losses.append(loss_i)

    return torch.stack(losses).mean()


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.encoder = AutoModel.from_pretrained(model_name).to(device)
        self.encoder.eval()

        for p in self.encoder.parameters():
            p.requires_grad = False

        self.embed_dim = int(self.encoder.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.embed_dim, dtype=torch.float32)

        inputs = self.tokenizer(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device, non_blocking=True) for k, v in inputs.items()}

        use_amp = (self.device.type == "cuda")
        if self.device.type == "cuda":
            dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
            with amp.autocast(device_type="cuda", enabled=use_amp, dtype=dtype):
                out = self.encoder(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.encoder(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _text_to_chunk_texts(self, text: str) -> List[str]:
        ids = self.tokenizer(text, add_special_tokens=False)["input_ids"]
        if len(ids) == 0:
            return []

        chunk_texts = []
        for i in range(0, len(ids), self.chunk_tok_len):
            chunk_ids = ids[i:i + self.chunk_tok_len]
            chunk_text = self.tokenizer.decode(chunk_ids, clean_up_tokenization_spaces=True)
            chunk_texts.append(chunk_text)
        return chunk_texts

    def encode_text(self, text: str) -> torch.Tensor:
        chunk_texts = self._text_to_chunk_texts(text)
        if len(chunk_texts) == 0:
            return torch.zeros(self.seq_len, self.embed_dim, dtype=torch.float32)

        if len(chunk_texts) > self.seq_len:
            chunk_texts = chunk_texts[:self.seq_len]

        out = []
        for i in range(0, len(chunk_texts), self.batch_size):
            batch = chunk_texts[i:i + self.batch_size]
            out.append(self._embed_chunk_texts(batch))

        embs = torch.cat(out, dim=0)
        t = embs.size(0)

        if t < self.seq_len:
            pad = torch.zeros(self.seq_len - t, embs.size(1), dtype=embs.dtype)
            embs = torch.cat([embs, pad], dim=0)

        return embs[:self.seq_len]

    def encode_choice_set(self, question_stem: str, choice_texts: List[str]) -> torch.Tensor:
        feats = []
        for ch in choice_texts:
            text = f"Question: {question_stem}\nAnswer Choice: {ch}"
            feats.append(self.encode_text(text))
        return torch.stack(feats, dim=0)


# ============================================================
# DATA PREP
# ============================================================

def load_raw_mcq_dataset(dataset_name: str):
    if dataset_name == "ARC-Easy":
        return load_dataset("allenai/ai2_arc", "ARC-Easy")
    if dataset_name == "ARC-Challenge":
        return load_dataset("allenai/ai2_arc", "ARC-Challenge")
    raise ValueError(f"Unknown dataset: {dataset_name}")


def normalize_choices_field(example: Dict[str, Any]) -> Tuple[str, List[str], int]:
    """
    ARC format:
      question: string
      choices: {"label": [...], "text": [...]}
      answerKey: string
    Example:
      {
        "answerKey": "B",
        "choices": {
            "label": ["A","B","C","D"],
            "text":  ["...","...","...","..."]
        },
        "id": "...",
        "question": "..."
      }
    """
    if "question" not in example:
        raise KeyError("question")
    stem = str(example["question"])

    if "choices" not in example:
        raise KeyError("choices")
    choices = example["choices"]

    if not isinstance(choices, dict):
        raise TypeError(f"Expected choices to be dict, got {type(choices)}")
    if "text" not in choices:
        raise KeyError("choices.text")

    choice_texts = list(choices["text"])

    if "answerKey" not in example:
        raise KeyError("answerKey")
    label = answerkey_to_index(example["answerKey"], len(choice_texts))

    return stem, choice_texts, label


def build_or_load_cached_split(
    dataset_name: str,
    split_name: str,
    split_data,
    conceptizer: DebertaConceptizer,
    cache_dir: str,
) -> List[Dict[str, Any]]:
    ensure_dir(cache_dir)
    safe_ds = dataset_name.replace("/", "_")
    path = os.path.join(
        cache_dir,
        f"{safe_ds}_{split_name}_seq{conceptizer.seq_len}_tok{conceptizer.chunk_tok_len}.pt"
    )

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    rows = []
    skipped = 0
    print(f"[cache] building {dataset_name} / {split_name}")

    for ex in tqdm(split_data, desc=f"Conceptizing {dataset_name}-{split_name}"):
        try:
            stem, choice_texts, label = normalize_choices_field(ex)
            x = conceptizer.encode_choice_set(stem, choice_texts)
            choice_mask = torch.ones(x.size(0), dtype=torch.bool)

            rows.append({
                "x": x,
                "label": int(label),
                "choice_mask": choice_mask,
                "num_choices": int(x.size(0)),
            })
        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example in {dataset_name}-{split_name}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")
    return rows


class MCQFeatureDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx: int):
        r = self.rows[idx]
        return {
            "x": r["x"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "choice_mask": r["choice_mask"],
            "num_choices": r["num_choices"],
        }


def mcq_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    max_c = max(item["x"].size(0) for item in batch)
    t = batch[0]["x"].size(1)
    d = batch[0]["x"].size(2)
    b = len(batch)

    x = torch.zeros(b, max_c, t, d, dtype=torch.float32)
    choice_mask = torch.zeros(b, max_c, dtype=torch.bool)
    labels = torch.zeros(b, dtype=torch.long)

    for i, item in enumerate(batch):
        c = item["x"].size(0)
        x[i, :c] = item["x"]
        choice_mask[i, :c] = item["choice_mask"]
        labels[i] = item["label"]

    return {
        "x": x,
        "choice_mask": choice_mask,
        "labels": labels,
    }


# ============================================================
# MODEL
# ============================================================

class MCQHead(nn.Module):
    """
    Frozen HLCM + LayerNorm-stabilized classifier head.
    """
    def __init__(self, model: HyperbolicLCM, dropout: float = 0.5):
        super().__init__()
        self.model = model
        dim = model.layers[0].attn.embed_dim

        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dim, 1)

        nn.init.normal_(self.classifier.weight, mean=0.0, std=0.01)
        nn.init.zeros_(self.classifier.bias)

    def forward(
        self,
        x: torch.Tensor,
        choice_mask: torch.Tensor,
        mu: Optional[torch.Tensor] = None,
        sigma: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        b, c, t, din = x.shape
        x = x.reshape(b * c, t, din)

        if mu is not None and sigma is not None:
            x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

        with torch.no_grad():
            h = self.model(x)

        last_h = h[:, -1, :]
        last_tan = self.model.manifold.logmap0(last_h)
        last_tan = self.norm(last_tan)
        last_tan = self.dropout(last_tan)

        logits = self.classifier(last_tan).view(b, c)
        logits = logits.masked_fill(~choice_mask, -1e9)
        return logits


def build_hlcm_from_cfg(cfg: FinetuneConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: FinetuneConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"[load] loaded HLCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    model.eval()
    for p in model.parameters():
        p.requires_grad = False

    return model


# ============================================================
# OPTIM / SCHED
# ============================================================

class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps: int, total_steps: int, min_lr_scale: float = 0.1):
        self.optimizer = optimizer
        self.warmup_steps = max(0, int(warmup_steps))
        self.total_steps = max(1, int(total_steps))
        self.min_lr_scale = float(min_lr_scale)
        self.step_num = 0
        self.base_lrs = [pg["lr"] for pg in optimizer.param_groups]

    def step(self):
        self.step_num += 1
        s = self.step_num

        if s <= self.warmup_steps and self.warmup_steps > 0:
            scale = s / self.warmup_steps
        else:
            denom = max(1, self.total_steps - self.warmup_steps)
            p = min(1.0, max(0.0, (s - self.warmup_steps) / denom))
            cosine = 0.5 * (1.0 + math.cos(math.pi * p))
            scale = self.min_lr_scale + (1.0 - self.min_lr_scale) * cosine

        for lr0, pg in zip(self.base_lrs, self.optimizer.param_groups):
            pg["lr"] = lr0 * scale


# ============================================================
# TRAIN / EVAL
# ============================================================

@torch.no_grad()
def evaluate(
    model: MCQHead,
    loader: DataLoader,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> Dict[str, float]:
    model.eval()

    total_loss = 0.0
    total = 0
    correct = 0

    for batch in loader:
        x = batch["x"].to(device, non_blocking=True)
        choice_mask = batch["choice_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        logits = model(x, choice_mask, mu=mu, sigma=sigma)
        loss = masked_choice_cross_entropy(
            logits=logits,
            labels=labels,
            choice_mask=choice_mask,
            label_smoothing=0.0,
        )

        total_loss += float(loss.item()) * x.size(0)
        preds = logits.argmax(dim=-1)
        correct += int((preds == labels).sum().item())
        total += int(x.size(0))

    model.train()
    return {
        "loss": total_loss / max(total, 1),
        "acc": correct / max(total, 1),
    }


def train_one_dataset(dataset_name: str, cfg: FinetuneConfig):
    print(f"\n==================== {dataset_name} ====================")
    device = pick_device(cfg.prefer_gpu_index)
    set_seed(cfg.seed)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    out_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    ensure_dir(out_dir)
    ensure_dir(cfg.cache_dir)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=device,
    )

    raw_ds = load_raw_mcq_dataset(dataset_name)

    train_rows = build_or_load_cached_split(
        dataset_name, "train", raw_ds["train"], conceptizer, cfg.cache_dir
    )

    val_rows = build_or_load_cached_split(
        dataset_name, "validation", raw_ds["validation"], conceptizer, cfg.cache_dir
    )

    test_rows = build_or_load_cached_split(
        dataset_name, "test", raw_ds["test"], conceptizer, cfg.cache_dir
    )

    if len(train_rows) == 0:
        raise RuntimeError(f"No usable training rows for {dataset_name}")
    if len(val_rows) == 0:
        raise RuntimeError(f"No usable validation rows for {dataset_name}")

    train_ds = MCQFeatureDataset(train_rows)
    val_ds = MCQFeatureDataset(val_rows)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    test_loader = None
    if len(test_rows) > 0:
        test_ds = MCQFeatureDataset(test_rows)
        test_loader = DataLoader(
            test_ds,
            batch_size=cfg.eval_batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=(device.type == "cuda"),
            collate_fn=mcq_collate,
            drop_last=False,
        )

    hlcm = load_pretrained_hlcm(cfg, device)
    model = MCQHead(hlcm, dropout=cfg.head_dropout).to(device)

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        [{"params": trainable_params, "lr": cfg.head_lr, "weight_decay": cfg.weight_decay}]
    )

    steps_per_epoch = math.ceil(len(train_loader) / max(cfg.grad_accum_steps, 1))
    total_steps = max(1, cfg.epochs * steps_per_epoch)
    warmup_steps = int(cfg.warmup_ratio * total_steps)
    scheduler = WarmupCosineScheduler(
        optimizer,
        warmup_steps=warmup_steps,
        total_steps=total_steps,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"
    amp_dtype = torch.bfloat16

    history = []
    best_val_acc = -1.0
    best_path = os.path.join(out_dir, "best_mcq.pt")

    start_time = time.time()

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        running_loss = 0.0
        seen = 0

        pbar = tqdm(train_loader, desc=f"Train {dataset_name} epoch {epoch}/{cfg.epochs}", mininterval=1.0)

        for step, batch in enumerate(pbar, start=1):
            x = batch["x"].to(device, non_blocking=True)
            choice_mask = batch["choice_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=amp_dtype):
                    logits = model(x, choice_mask, mu=mu, sigma=sigma)
                    loss = masked_choice_cross_entropy(
                        logits=logits,
                        labels=labels,
                        choice_mask=choice_mask,
                        label_smoothing=float(cfg.label_smoothing),
                    )
            else:
                logits = model(x, choice_mask, mu=mu, sigma=sigma)
                loss = masked_choice_cross_entropy(
                    logits=logits,
                    labels=labels,
                    choice_mask=choice_mask,
                    label_smoothing=float(cfg.label_smoothing),
                )

            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True)
                continue

            (loss / cfg.grad_accum_steps).backward()

            running_loss += float(loss.item()) * x.size(0)
            seen += int(x.size(0))

            if step % cfg.grad_accum_steps == 0 or step == len(train_loader):
                torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            pbar.set_postfix(loss=f"{running_loss / max(seen, 1):.4f}")

        train_loss = running_loss / max(seen, 1)
        val_metrics = evaluate(model, val_loader, device, mu, sigma)

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["acc"],
        }

        if test_loader is not None:
            test_metrics = evaluate(model, test_loader, device, mu, sigma)
            row["test_loss"] = test_metrics["loss"]
            row["test_acc"] = test_metrics["acc"]
        else:
            test_metrics = None

        history.append(row)

        print(
            f"[epoch {epoch}] "
            f"train_loss={train_loss:.4f} "
            f"val_loss={val_metrics['loss']:.4f} "
            f"val_acc={val_metrics['acc']:.4f}"
            + (f" test_acc={test_metrics['acc']:.4f}" if test_metrics is not None else "")
        )

        if val_metrics["acc"] > best_val_acc:
            best_val_acc = float(val_metrics["acc"])
            torch.save(
                {
                    "cfg": asdict(cfg),
                    "dataset_name": dataset_name,
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "hlcm_state": model.model.state_dict(),
                    "best_val_acc": best_val_acc,
                    "history": history,
                },
                best_path,
            )
            print(f"[save] best checkpoint -> {best_path}")

    wall = time.time() - start_time

    best_obj = torch.load(best_path, map_location=device)
    model.load_state_dict(best_obj["model_state"], strict=True)

    final_val = evaluate(model, val_loader, device, mu, sigma)
    final_test = evaluate(model, test_loader, device, mu, sigma) if test_loader is not None else None

    summary = {
        "dataset_name": dataset_name,
        "best_val_acc": float(final_val["acc"]),
        "best_val_loss": float(final_val["loss"]),
        "test_acc": None if final_test is None else float(final_test["acc"]),
        "test_loss": None if final_test is None else float(final_test["loss"]),
        "epochs": cfg.epochs,
        "wall_time_sec": float(wall),
        "wall_time_hms": fmt_hms(wall),
        "num_train_examples": len(train_ds),
        "num_val_examples": len(val_ds),
        "num_test_examples": len(test_rows),
        "history": history,
    }

    with open(os.path.join(out_dir, "summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print(f"[done] {dataset_name}")
    print(json.dumps(summary, indent=2))

    return summary


# ============================================================
# MAIN
# ============================================================

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--ckpt_path", type=str, default="runs/hyperbolic_cluster/checkpoints/ckpt_best.pt")
    p.add_argument("--normalizer_path", type=str, default="normalizer.pt")
    p.add_argument("--out_dir", type=str, default="runs/mcq_hlcm_arc_only")
    p.add_argument("--cache_dir", type=str, default="mcq_cache")

    p.add_argument("--epochs", type=int, default=3)
    p.add_argument("--train_batch_size", type=int, default=4)
    p.add_argument("--eval_batch_size", type=int, default=8)
    p.add_argument("--grad_accum_steps", type=int, default=1)

    p.add_argument("--lr", type=float, default=0.0)
    p.add_argument("--head_lr", type=float, default=3e-5)
    p.add_argument("--weight_decay", type=float, default=0.01)
    p.add_argument("--warmup_ratio", type=float, default=0.06)
    p.add_argument("--max_grad_norm", type=float, default=1.0)

    p.add_argument("--seq_len", type=int, default=8)
    p.add_argument("--chunk_tok_len", type=int, default=256)
    p.add_argument("--encoder_batch_size", type=int, default=64)

    p.add_argument("--prefer_gpu_index", type=int, default=0)
    p.add_argument("--seed", type=int, default=42)

    p.add_argument("--label_smoothing", type=float, default=0.10)

    if "ipykernel" in sys.modules:
        return p.parse_args(args=[])

    return p.parse_args()


def main():
    args = parse_args()

    cfg = FinetuneConfig(
        ckpt_path=args.ckpt_path,
        normalizer_path=args.normalizer_path,
        out_dir=args.out_dir,
        cache_dir=args.cache_dir,
        epochs=args.epochs,
        train_batch_size=args.train_batch_size,
        eval_batch_size=args.eval_batch_size,
        grad_accum_steps=args.grad_accum_steps,
        lr=args.lr,
        head_lr=args.head_lr,
        weight_decay=args.weight_decay,
        warmup_ratio=args.warmup_ratio,
        max_grad_norm=args.max_grad_norm,
        seq_len=args.seq_len,
        chunk_tok_len=args.chunk_tok_len,
        encoder_batch_size=args.encoder_batch_size,
        prefer_gpu_index=args.prefer_gpu_index,
        seed=args.seed,
        label_smoothing=args.label_smoothing,
        freeze_hlcm=True,
    )

    ensure_dir(cfg.out_dir)

    all_summaries = {}
    for ds_name in cfg.datasets_to_run:
        summary = train_one_dataset(ds_name, cfg)
        all_summaries[ds_name] = summary

    with open(os.path.join(cfg.out_dir, "all_results.json"), "w") as f:
        json.dump(all_summaries, f, indent=2)

    print("\n==================== ALL DONE ====================")
    print(json.dumps(all_summaries, indent=2))


if __name__ == "__main__":
    main()


==================== ARC-Easy ====================


/home/user/anaconda3/envs/raat/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


[cache] building ARC-Easy / train


Conceptizing ARC-Easy-train: 100%|███████████████████████████████████████████████████████████| 2251/2251 [00:48<00:00, 46.31it/s]


[cache] saved mcq_cache/ARC-Easy_train_seq8_tok256.pt (2251 examples, skipped=0)
[cache] building ARC-Easy / validation


...

[epoch 1] train_loss=1.6147 val_loss=1.3840 val_acc=0.2508 test_acc=0.2765
[save] best checkpoint -> runs/mcq_hlcm_arc_only/ARC-Challenge/best_mcq.pt


Train ARC-Challenge epoch 2/3: 100%|██████████████████████████████████████████████| 280/280 [00:22<00:00, 12.72it/s, loss=1.6088]


[epoch 2] train_loss=1.6088 val_loss=1.3839 val_acc=0.2475 test_acc=0.2782


Train ARC-Challenge epoch 3/3: 100%|██████████████████████████████████████████████| 280/280 [00:21<00:00, 13.25it/s, loss=1.6330]


[epoch 3] train_loss=1.6330 val_loss=1.3839 val_acc=0.2475 test_acc=0.2765
[done] ARC-Challenge
{
  "dataset_name": "ARC-Challenge",
  "best_val_acc": 0.2508361204013378,
  "best_val_loss": 1.3839548491315299,
  "test_acc": 0.2764505119453925,
  "test_loss": 1.3839418497508704,
  "epochs": 3,
  "wall_time_sec": 113.95876169204712,
  "wall_time_hms": "00:01:53",
  "num_train_examples": 1119,
  "num_val_examples": 299,
  "num_test_examples": 1172,
  "history": [
    {
      "epoch": 1,
      "train_loss": 1.6147472660279465,
      "val_loss": 1.3839549771120716,
      "val_acc": 0.2508361204013378,
      "test_loss": 1.3839418497508704,
      "test_acc": 0.2764505119453925
    },
    {
      "epoch": 2,
      "train_loss": 1.6088065291431144,
      "val_loss": 1.3839216088770225,
      "val_acc": 0.24749163879598662,
      "test_loss": 1.3839057708356162,
      "test_acc": 0.2781569965870307
    },
    {
      "epoch": 3,
      "train_loss": 1.6329906310643119,
      "val_loss": 1.383901

In [1]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import sys
import math
import json
import time
import random
import argparse
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class FinetuneConfig:
    datasets_to_run: Tuple[str, ...] = ("ARC-Easy", "ARC-Challenge")
    cache_dir: str = "mcq_cache"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 64

    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    epochs: int = 3
    train_batch_size: int = 4
    eval_batch_size: int = 8
    grad_accum_steps: int = 1
    lr: float = 0.0
    head_lr: float = 3e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    max_grad_norm: float = 1.0
    label_smoothing: float = 0.10

    freeze_encoder: bool = True
    freeze_hlcm: bool = True

    head_dropout: float = 0.50

    seed: int = 42
    num_workers: int = 4
    use_bf16: bool = True
    prefer_gpu_index: int = 0

    out_dir: str = "runs/mcq_hlcm_arc_only"


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds: float) -> str:
    seconds = max(float(seconds), 0.0)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def answerkey_to_index(answer_key: str, num_choices: int) -> int:
    answer_key = str(answer_key).strip()
    alpha = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
    numeric = {"1": 0, "2": 1, "3": 2, "4": 3, "5": 4}

    if answer_key in alpha:
        idx = alpha[answer_key]
    elif answer_key in numeric:
        idx = numeric[answer_key]
    else:
        raise ValueError(f"Unknown answerKey: {answer_key}")

    if idx >= num_choices:
        raise ValueError(f"answerKey={answer_key} -> idx={idx}, but num_choices={num_choices}")
    return idx


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None
    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


# ============================================================
# MASKED LOSS
# ============================================================

def masked_choice_cross_entropy(
    logits: torch.Tensor,         # [B, C]
    labels: torch.Tensor,         # [B]
    choice_mask: torch.Tensor,    # [B, C] bool
    label_smoothing: float = 0.0,
) -> torch.Tensor:
    """
    Cross-entropy over only valid choices.
    Label smoothing mass is distributed only over valid choices,
    never onto padded choices.
    """
    b, _ = logits.shape
    log_probs = F.log_softmax(logits, dim=-1)

    losses = []
    for i in range(b):
        valid = choice_mask[i]
        valid_idx = torch.nonzero(valid, as_tuple=False).squeeze(-1)
        lp = log_probs[i, valid]  # [Cv]

        gold_global = int(labels[i].item())
        local_pos = (valid_idx == gold_global).nonzero(as_tuple=False)
        if local_pos.numel() == 0:
            raise RuntimeError(f"Gold label {gold_global} not found among valid choices in row {i}")
        local_pos = int(local_pos.item())

        cv = int(lp.size(0))
        if label_smoothing > 0.0:
            target = torch.full_like(lp, fill_value=label_smoothing / cv)
            target[local_pos] += (1.0 - label_smoothing)
            loss_i = -(target * lp).sum()
        else:
            loss_i = -lp[local_pos]

        losses.append(loss_i)

    return torch.stack(losses).mean()


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.encoder = AutoModel.from_pretrained(model_name).to(device)
        self.encoder.eval()

        for p in self.encoder.parameters():
            p.requires_grad = False

        self.embed_dim = int(self.encoder.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.embed_dim, dtype=torch.float32)

        inputs = self.tokenizer(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device, non_blocking=True) for k, v in inputs.items()}

        use_amp = (self.device.type == "cuda")
        if self.device.type == "cuda":
            dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
            with amp.autocast(device_type="cuda", enabled=use_amp, dtype=dtype):
                out = self.encoder(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.encoder(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _text_to_chunk_texts(self, text: str) -> List[str]:
        ids = self.tokenizer(text, add_special_tokens=False)["input_ids"]
        if len(ids) == 0:
            return []

        chunk_texts = []
        for i in range(0, len(ids), self.chunk_tok_len):
            chunk_ids = ids[i:i + self.chunk_tok_len]
            chunk_text = self.tokenizer.decode(chunk_ids, clean_up_tokenization_spaces=True)
            chunk_texts.append(chunk_text)
        return chunk_texts

    def encode_text(self, text: str) -> torch.Tensor:
        chunk_texts = self._text_to_chunk_texts(text)
        if len(chunk_texts) == 0:
            return torch.zeros(self.seq_len, self.embed_dim, dtype=torch.float32)

        if len(chunk_texts) > self.seq_len:
            chunk_texts = chunk_texts[:self.seq_len]

        out = []
        for i in range(0, len(chunk_texts), self.batch_size):
            batch = chunk_texts[i:i + self.batch_size]
            out.append(self._embed_chunk_texts(batch))

        embs = torch.cat(out, dim=0)
        t = embs.size(0)

        if t < self.seq_len:
            pad = torch.zeros(self.seq_len - t, embs.size(1), dtype=embs.dtype)
            embs = torch.cat([embs, pad], dim=0)

        return embs[:self.seq_len]

    def encode_choice_set(self, question_stem: str, choice_texts: List[str]) -> torch.Tensor:
        feats = []
        for ch in choice_texts:
            text = f"Question: {question_stem}\nAnswer Choice: {ch}"
            feats.append(self.encode_text(text))
        return torch.stack(feats, dim=0)


# ============================================================
# DATA PREP
# ============================================================

def load_raw_mcq_dataset(dataset_name: str):
    if dataset_name == "ARC-Easy":
        return load_dataset("allenai/ai2_arc", "ARC-Easy")
    if dataset_name == "ARC-Challenge":
        return load_dataset("allenai/ai2_arc", "ARC-Challenge")
    raise ValueError(f"Unknown dataset: {dataset_name}")


def normalize_choices_field(example: Dict[str, Any]) -> Tuple[str, List[str], int]:
    """
    ARC format:
      question: string
      choices: {"label": [...], "text": [...]}
      answerKey: string
    Example:
      {
        "answerKey": "B",
        "choices": {
            "label": ["A","B","C","D"],
            "text":  ["...","...","...","..."]
        },
        "id": "...",
        "question": "..."
      }
    """
    if "question" not in example:
        raise KeyError("question")
    stem = str(example["question"])

    if "choices" not in example:
        raise KeyError("choices")
    choices = example["choices"]

    if not isinstance(choices, dict):
        raise TypeError(f"Expected choices to be dict, got {type(choices)}")
    if "text" not in choices:
        raise KeyError("choices.text")

    choice_texts = list(choices["text"])

    if "answerKey" not in example:
        raise KeyError("answerKey")
    label = answerkey_to_index(example["answerKey"], len(choice_texts))

    return stem, choice_texts, label


def build_or_load_cached_split(
    dataset_name: str,
    split_name: str,
    split_data,
    conceptizer: DebertaConceptizer,
    cache_dir: str,
) -> List[Dict[str, Any]]:
    ensure_dir(cache_dir)
    safe_ds = dataset_name.replace("/", "_")
    path = os.path.join(
        cache_dir,
        f"{safe_ds}_{split_name}_seq{conceptizer.seq_len}_tok{conceptizer.chunk_tok_len}.pt"
    )

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    rows = []
    skipped = 0
    print(f"[cache] building {dataset_name} / {split_name}")

    for ex in tqdm(split_data, desc=f"Conceptizing {dataset_name}-{split_name}"):
        try:
            stem, choice_texts, label = normalize_choices_field(ex)
            x = conceptizer.encode_choice_set(stem, choice_texts)
            choice_mask = torch.ones(x.size(0), dtype=torch.bool)

            rows.append({
                "x": x,
                "label": int(label),
                "choice_mask": choice_mask,
                "num_choices": int(x.size(0)),
            })
        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example in {dataset_name}-{split_name}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")
    return rows


class MCQFeatureDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx: int):
        r = self.rows[idx]
        return {
            "x": r["x"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "choice_mask": r["choice_mask"],
            "num_choices": r["num_choices"],
        }


def mcq_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    max_c = max(item["x"].size(0) for item in batch)
    t = batch[0]["x"].size(1)
    d = batch[0]["x"].size(2)
    b = len(batch)

    x = torch.zeros(b, max_c, t, d, dtype=torch.float32)
    choice_mask = torch.zeros(b, max_c, dtype=torch.bool)
    labels = torch.zeros(b, dtype=torch.long)

    for i, item in enumerate(batch):
        c = item["x"].size(0)
        x[i, :c] = item["x"]
        choice_mask[i, :c] = item["choice_mask"]
        labels[i] = item["label"]

    return {
        "x": x,
        "choice_mask": choice_mask,
        "labels": labels,
    }


# ============================================================
# MODEL
# ============================================================

class MCQHead(nn.Module):
    """
    Frozen HLCM + LayerNorm-stabilized classifier head.
    """
    def __init__(self, model: HyperbolicLCM, dropout: float = 0.5):
        super().__init__()
        self.model = model
        dim = model.layers[0].attn.embed_dim

        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dim, 1)

        nn.init.normal_(self.classifier.weight, mean=0.0, std=0.01)
        nn.init.zeros_(self.classifier.bias)

    def forward(
        self,
        x: torch.Tensor,
        choice_mask: torch.Tensor,
        mu: Optional[torch.Tensor] = None,
        sigma: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        b, c, t, din = x.shape
        x = x.reshape(b * c, t, din)

        if mu is not None and sigma is not None:
            x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

        with torch.no_grad():
            h = self.model(x)

        last_h = h[:, -1, :]
        last_tan = self.model.manifold.logmap0(last_h)
        last_tan = self.norm(last_tan)
        last_tan = self.dropout(last_tan)

        logits = self.classifier(last_tan).view(b, c)
        logits = logits.masked_fill(~choice_mask, -1e9)
        return logits


def build_hlcm_from_cfg(cfg: FinetuneConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: FinetuneConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"[load] loaded HLCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    model.eval()
    for p in model.parameters():
        p.requires_grad = False

    return model


# ============================================================
# OPTIM / SCHED
# ============================================================

class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps: int, total_steps: int, min_lr_scale: float = 0.1):
        self.optimizer = optimizer
        self.warmup_steps = max(0, int(warmup_steps))
        self.total_steps = max(1, int(total_steps))
        self.min_lr_scale = float(min_lr_scale)
        self.step_num = 0
        self.base_lrs = [pg["lr"] for pg in optimizer.param_groups]

    def step(self):
        self.step_num += 1
        s = self.step_num

        if s <= self.warmup_steps and self.warmup_steps > 0:
            scale = s / self.warmup_steps
        else:
            denom = max(1, self.total_steps - self.warmup_steps)
            p = min(1.0, max(0.0, (s - self.warmup_steps) / denom))
            cosine = 0.5 * (1.0 + math.cos(math.pi * p))
            scale = self.min_lr_scale + (1.0 - self.min_lr_scale) * cosine

        for lr0, pg in zip(self.base_lrs, self.optimizer.param_groups):
            pg["lr"] = lr0 * scale


# ============================================================
# TRAIN / EVAL
# ============================================================

@torch.no_grad()
def evaluate(
    model: MCQHead,
    loader: DataLoader,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> Dict[str, float]:
    model.eval()

    total_loss = 0.0
    total = 0
    correct = 0

    for batch in loader:
        x = batch["x"].to(device, non_blocking=True)
        choice_mask = batch["choice_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        logits = model(x, choice_mask, mu=mu, sigma=sigma)
        loss = masked_choice_cross_entropy(
            logits=logits,
            labels=labels,
            choice_mask=choice_mask,
            label_smoothing=0.0,
        )

        total_loss += float(loss.item()) * x.size(0)
        preds = logits.argmax(dim=-1)
        correct += int((preds == labels).sum().item())
        total += int(x.size(0))

    model.train()
    return {
        "loss": total_loss / max(total, 1),
        "acc": correct / max(total, 1),
    }


def train_one_dataset(dataset_name: str, cfg: FinetuneConfig):
    print(f"\n==================== {dataset_name} ====================")
    device = pick_device(cfg.prefer_gpu_index)
    set_seed(cfg.seed)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    out_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    ensure_dir(out_dir)
    ensure_dir(cfg.cache_dir)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=device,
    )

    raw_ds = load_raw_mcq_dataset(dataset_name)

    train_rows = build_or_load_cached_split(
        dataset_name, "train", raw_ds["train"], conceptizer, cfg.cache_dir
    )

    val_rows = build_or_load_cached_split(
        dataset_name, "validation", raw_ds["validation"], conceptizer, cfg.cache_dir
    )

    test_rows = build_or_load_cached_split(
        dataset_name, "test", raw_ds["test"], conceptizer, cfg.cache_dir
    )

    if len(train_rows) == 0:
        raise RuntimeError(f"No usable training rows for {dataset_name}")
    if len(val_rows) == 0:
        raise RuntimeError(f"No usable validation rows for {dataset_name}")

    train_ds = MCQFeatureDataset(train_rows)
    val_ds = MCQFeatureDataset(val_rows)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    test_loader = None
    if len(test_rows) > 0:
        test_ds = MCQFeatureDataset(test_rows)
        test_loader = DataLoader(
            test_ds,
            batch_size=cfg.eval_batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=(device.type == "cuda"),
            collate_fn=mcq_collate,
            drop_last=False,
        )

    hlcm = load_pretrained_hlcm(cfg, device)
    model = MCQHead(hlcm, dropout=cfg.head_dropout).to(device)

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        [{"params": trainable_params, "lr": cfg.head_lr, "weight_decay": cfg.weight_decay}]
    )

    steps_per_epoch = math.ceil(len(train_loader) / max(cfg.grad_accum_steps, 1))
    total_steps = max(1, cfg.epochs * steps_per_epoch)
    warmup_steps = int(cfg.warmup_ratio * total_steps)
    scheduler = WarmupCosineScheduler(
        optimizer,
        warmup_steps=warmup_steps,
        total_steps=total_steps,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"
    amp_dtype = torch.bfloat16

    history = []
    best_val_acc = -1.0
    best_path = os.path.join(out_dir, "best_mcq.pt")

    start_time = time.time()

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        running_loss = 0.0
        seen = 0

        pbar = tqdm(train_loader, desc=f"Train {dataset_name} epoch {epoch}/{cfg.epochs}", mininterval=1.0)

        for step, batch in enumerate(pbar, start=1):
            x = batch["x"].to(device, non_blocking=True)
            choice_mask = batch["choice_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=amp_dtype):
                    logits = model(x, choice_mask, mu=mu, sigma=sigma)
                    loss = masked_choice_cross_entropy(
                        logits=logits,
                        labels=labels,
                        choice_mask=choice_mask,
                        label_smoothing=float(cfg.label_smoothing),
                    )
            else:
                logits = model(x, choice_mask, mu=mu, sigma=sigma)
                loss = masked_choice_cross_entropy(
                    logits=logits,
                    labels=labels,
                    choice_mask=choice_mask,
                    label_smoothing=float(cfg.label_smoothing),
                )

            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True)
                continue

            (loss / cfg.grad_accum_steps).backward()

            running_loss += float(loss.item()) * x.size(0)
            seen += int(x.size(0))

            if step % cfg.grad_accum_steps == 0 or step == len(train_loader):
                torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            pbar.set_postfix(loss=f"{running_loss / max(seen, 1):.4f}")

        train_loss = running_loss / max(seen, 1)
        val_metrics = evaluate(model, val_loader, device, mu, sigma)

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["acc"],
        }

        if test_loader is not None:
            test_metrics = evaluate(model, test_loader, device, mu, sigma)
            row["test_loss"] = test_metrics["loss"]
            row["test_acc"] = test_metrics["acc"]
        else:
            test_metrics = None

        history.append(row)

        print(
            f"[epoch {epoch}] "
            f"train_loss={train_loss:.4f} "
            f"val_loss={val_metrics['loss']:.4f} "
            f"val_acc={val_metrics['acc']:.4f}"
            + (f" test_acc={test_metrics['acc']:.4f}" if test_metrics is not None else "")
        )

        if val_metrics["acc"] > best_val_acc:
            best_val_acc = float(val_metrics["acc"])
            torch.save(
                {
                    "cfg": asdict(cfg),
                    "dataset_name": dataset_name,
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "hlcm_state": model.model.state_dict(),
                    "best_val_acc": best_val_acc,
                    "history": history,
                },
                best_path,
            )
            print(f"[save] best checkpoint -> {best_path}")

    wall = time.time() - start_time

    best_obj = torch.load(best_path, map_location=device)
    model.load_state_dict(best_obj["model_state"], strict=True)

    final_val = evaluate(model, val_loader, device, mu, sigma)
    final_test = evaluate(model, test_loader, device, mu, sigma) if test_loader is not None else None

    summary = {
        "dataset_name": dataset_name,
        "best_val_acc": float(final_val["acc"]),
        "best_val_loss": float(final_val["loss"]),
        "test_acc": None if final_test is None else float(final_test["acc"]),
        "test_loss": None if final_test is None else float(final_test["loss"]),
        "epochs": cfg.epochs,
        "wall_time_sec": float(wall),
        "wall_time_hms": fmt_hms(wall),
        "num_train_examples": len(train_ds),
        "num_val_examples": len(val_ds),
        "num_test_examples": len(test_rows),
        "history": history,
    }

    with open(os.path.join(out_dir, "summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print(f"[done] {dataset_name}")
    print(json.dumps(summary, indent=2))

    return summary


# ============================================================
# MAIN
# ============================================================

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--ckpt_path", type=str, default="runs/hyperbolic_cluster/checkpoints/ckpt_best.pt")
    p.add_argument("--normalizer_path", type=str, default="normalizer.pt")
    p.add_argument("--out_dir", type=str, default="runs/mcq_hlcm_arc_only")
    p.add_argument("--cache_dir", type=str, default="mcq_cache")

    p.add_argument("--epochs", type=int, default=3)
    p.add_argument("--train_batch_size", type=int, default=4)
    p.add_argument("--eval_batch_size", type=int, default=8)
    p.add_argument("--grad_accum_steps", type=int, default=1)

    p.add_argument("--lr", type=float, default=0.0)
    p.add_argument("--head_lr", type=float, default=3e-5)
    p.add_argument("--weight_decay", type=float, default=0.01)
    p.add_argument("--warmup_ratio", type=float, default=0.06)
    p.add_argument("--max_grad_norm", type=float, default=1.0)

    p.add_argument("--seq_len", type=int, default=8)
    p.add_argument("--chunk_tok_len", type=int, default=256)
    p.add_argument("--encoder_batch_size", type=int, default=64)

    p.add_argument("--prefer_gpu_index", type=int, default=0)
    p.add_argument("--seed", type=int, default=42)

    p.add_argument("--label_smoothing", type=float, default=0.10)

    if "ipykernel" in sys.modules:
        return p.parse_args(args=[])

    return p.parse_args()


def main():
    args = parse_args()

    cfg = FinetuneConfig(
        ckpt_path=args.ckpt_path,
        normalizer_path=args.normalizer_path,
        out_dir=args.out_dir,
        cache_dir=args.cache_dir,
        epochs=args.epochs,
        train_batch_size=args.train_batch_size,
        eval_batch_size=args.eval_batch_size,
        grad_accum_steps=args.grad_accum_steps,
        lr=args.lr,
        head_lr=args.head_lr,
        weight_decay=args.weight_decay,
        warmup_ratio=args.warmup_ratio,
        max_grad_norm=args.max_grad_norm,
        seq_len=args.seq_len,
        chunk_tok_len=args.chunk_tok_len,
        encoder_batch_size=args.encoder_batch_size,
        prefer_gpu_index=args.prefer_gpu_index,
        seed=args.seed,
        label_smoothing=args.label_smoothing,
        freeze_hlcm=True,
    )

    ensure_dir(cfg.out_dir)

    all_summaries = {}
    for ds_name in cfg.datasets_to_run:
        summary = train_one_dataset(ds_name, cfg)
        all_summaries[ds_name] = summary

    with open(os.path.join(cfg.out_dir, "all_results.json"), "w") as f:
        json.dump(all_summaries, f, indent=2)

    print("\n==================== ALL DONE ====================")
    print(json.dumps(all_summaries, indent=2))


# IMPORTANT:
# Commented out so training does NOT start again.
# if __name__ == "__main__":
#     main()

import time

@torch.inference_mode()
def infer_mcq(question, choices, best_mcq_path):
    cfg = FinetuneConfig()
    device = pick_device(cfg.prefer_gpu_index)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=device,
    )

    hlcm = load_pretrained_hlcm(cfg, device)
    model = MCQHead(hlcm, dropout=cfg.head_dropout).to(device)

    ckpt = torch.load(best_mcq_path, map_location=device)
    model.load_state_dict(ckpt["model_state"], strict=True)
    model.eval()

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    # -----------------------------
    # START TIMER
    # -----------------------------
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.time()

    # Encode
    x = conceptizer.encode_choice_set(question, choices)
    x = x.unsqueeze(0).to(device)

    choice_mask = torch.ones(
        1,
        len(choices),
        dtype=torch.bool,
        device=device,
    )

    # Forward
    logits = model(x, choice_mask, mu=mu, sigma=sigma)
    probs = torch.softmax(logits, dim=-1).squeeze(0)

    pred_idx = int(probs.argmax().item())

    # -----------------------------
    # END TIMER
    # -----------------------------
    if device.type == "cuda":
        torch.cuda.synchronize()
    end = time.time()

    inference_time = end - start

    return {
        "answer_index": pred_idx,
        "answer": choices[pred_idx],
        "probabilities": probs.cpu().tolist(),
        "inference_time_sec": inference_time,
    }

In [2]:
result = infer_mcq(
    question="Which planet is known as the Red Planet?",
    choices=["Earth", "Mars", "Jupiter", "Venus"],
    best_mcq_path="runs/mcq_hlcm_arc_only/ARC-Easy/best_mcq.pt",
)

print(result)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[load] loaded HLCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] missing keys: 0
[load] unexpected keys: 0
{'answer_index': 1, 'answer': 'Mars', 'probabilities': [0.248340904712677, 0.2554096579551697, 0.24843847751617432, 0.2478109896183014], 'inference_time_sec': 6.397550821304321}


In [2]:
result = infer_mcq(
    question="Which planet is known as the Red Planet?",
    choices=["Earth", "Mars", "Jupiter", "Venus"],
    best_mcq_path="runs/mcq_hlcm_arc_only/ARC-Challenge/best_mcq.pt",
)

print(result)

/home/user/anaconda3/envs/raat/lib/python3.10/site-packages/torch/cuda/__init__.py:174: UserWarning: CUDA initialization: CUDA driver initialization failed, you might not have a CUDA gpu. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[load] loaded HLCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] missing keys: 0
[load] unexpected keys: 0
{'answer_index': 1, 'answer': 'Mars', 'probabilities': [0.249126136302948, 0.25508683919906616, 0.2478373944759369, 0.24794970452785492], 'inference_time_sec': 25.31544589996338}


In [3]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import sys
import math
import json
import time
import random
import argparse
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class FinetuneConfig:
    datasets_to_run: Tuple[str, ...] = ("OpenBookQA",)
    cache_dir: str = "mcq_cache"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 64

    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    epochs: int = 3
    train_batch_size: int = 4
    eval_batch_size: int = 8
    grad_accum_steps: int = 1
    lr: float = 0.0
    head_lr: float = 3e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    max_grad_norm: float = 1.0
    label_smoothing: float = 0.10

    freeze_encoder: bool = True
    freeze_hlcm: bool = True

    head_dropout: float = 0.50

    seed: int = 42
    num_workers: int = 4
    use_bf16: bool = True
    prefer_gpu_index: int = 0

    out_dir: str = "runs/mcq_hlcm_openbookqa"


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds: float) -> str:
    seconds = max(float(seconds), 0.0)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def answerkey_to_index(answer_key: str, num_choices: int) -> int:
    answer_key = str(answer_key).strip()
    alpha = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
    numeric = {"1": 0, "2": 1, "3": 2, "4": 3, "5": 4}

    if answer_key in alpha:
        idx = alpha[answer_key]
    elif answer_key in numeric:
        idx = numeric[answer_key]
    else:
        raise ValueError(f"Unknown answerKey: {answer_key}")

    if idx >= num_choices:
        raise ValueError(f"answerKey={answer_key} -> idx={idx}, but num_choices={num_choices}")
    return idx


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None
    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


# ============================================================
# MASKED LOSS
# ============================================================

def masked_choice_cross_entropy(
    logits: torch.Tensor,         # [B, C]
    labels: torch.Tensor,         # [B]
    choice_mask: torch.Tensor,    # [B, C] bool
    label_smoothing: float = 0.0,
) -> torch.Tensor:
    """
    Cross-entropy over valid answer choices only.
    Label smoothing is distributed only across valid choices,
    not padded choices.
    """
    b, _ = logits.shape
    log_probs = F.log_softmax(logits, dim=-1)

    losses = []
    for i in range(b):
        valid = choice_mask[i]
        valid_idx = torch.nonzero(valid, as_tuple=False).squeeze(-1)  # [Cv]
        lp = log_probs[i, valid]  # [Cv]

        gold_global = int(labels[i].item())
        local_pos = (valid_idx == gold_global).nonzero(as_tuple=False)
        if local_pos.numel() == 0:
            raise RuntimeError(f"Gold label {gold_global} not found among valid choices for row {i}")
        local_pos = int(local_pos.item())

        cv = int(lp.size(0))
        if label_smoothing > 0.0:
            target = torch.full_like(lp, fill_value=label_smoothing / cv)
            target[local_pos] += (1.0 - label_smoothing)
            loss_i = -(target * lp).sum()
        else:
            loss_i = -lp[local_pos]

        losses.append(loss_i)

    return torch.stack(losses).mean()


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.encoder = AutoModel.from_pretrained(model_name).to(device)
        self.encoder.eval()

        for p in self.encoder.parameters():
            p.requires_grad = False

        self.embed_dim = int(self.encoder.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.embed_dim, dtype=torch.float32)

        inputs = self.tokenizer(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device, non_blocking=True) for k, v in inputs.items()}

        use_amp = (self.device.type == "cuda")
        if self.device.type == "cuda":
            dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
            with amp.autocast(device_type="cuda", enabled=use_amp, dtype=dtype):
                out = self.encoder(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.encoder(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _text_to_chunk_texts(self, text: str) -> List[str]:
        ids = self.tokenizer(text, add_special_tokens=False)["input_ids"]
        if len(ids) == 0:
            return []

        chunk_texts = []
        for i in range(0, len(ids), self.chunk_tok_len):
            chunk_ids = ids[i:i + self.chunk_tok_len]
            chunk_text = self.tokenizer.decode(chunk_ids, clean_up_tokenization_spaces=True)
            chunk_texts.append(chunk_text)
        return chunk_texts

    def encode_text(self, text: str) -> torch.Tensor:
        chunk_texts = self._text_to_chunk_texts(text)
        if len(chunk_texts) == 0:
            return torch.zeros(self.seq_len, self.embed_dim, dtype=torch.float32)

        if len(chunk_texts) > self.seq_len:
            chunk_texts = chunk_texts[:self.seq_len]

        out = []
        for i in range(0, len(chunk_texts), self.batch_size):
            batch = chunk_texts[i:i + self.batch_size]
            out.append(self._embed_chunk_texts(batch))

        embs = torch.cat(out, dim=0)
        t = embs.size(0)

        if t < self.seq_len:
            pad = torch.zeros(self.seq_len - t, embs.size(1), dtype=embs.dtype)
            embs = torch.cat([embs, pad], dim=0)

        return embs[:self.seq_len]

    def encode_choice_set(self, question_stem: str, choice_texts: List[str]) -> torch.Tensor:
        feats = []
        for ch in choice_texts:
            text = f"Question: {question_stem}\nAnswer Choice: {ch}"
            feats.append(self.encode_text(text))
        return torch.stack(feats, dim=0)


# ============================================================
# DATA PREP
# ============================================================

def load_raw_mcq_dataset(dataset_name: str):
    if dataset_name == "OpenBookQA":
        return load_dataset("allenai/openbookqa")
    raise ValueError(f"Unknown dataset: {dataset_name}")


def normalize_choices_field(example: Dict[str, Any]) -> Tuple[str, List[str], int]:
    """
    OpenBookQA format:
      {
        'id': '7-980',
        'question_stem': 'The sun is responsible for',
        'choices': {
            'text': [...],
            'label': ['A','B','C','D']
        },
        'answerKey': 'D'
      }
    """
    if "question_stem" not in example:
        raise KeyError("question_stem")
    stem = str(example["question_stem"])

    if "choices" not in example:
        raise KeyError("choices")
    choices = example["choices"]

    if not isinstance(choices, dict):
        raise TypeError(f"Expected choices to be dict, got {type(choices)}")
    if "text" not in choices:
        raise KeyError("choices.text")

    choice_texts = list(choices["text"])

    if "answerKey" not in example:
        raise KeyError("answerKey")
    label = answerkey_to_index(example["answerKey"], len(choice_texts))

    return stem, choice_texts, label


def build_or_load_cached_split(
    dataset_name: str,
    split_name: str,
    split_data,
    conceptizer: DebertaConceptizer,
    cache_dir: str,
) -> List[Dict[str, Any]]:
    ensure_dir(cache_dir)
    safe_ds = dataset_name.replace("/", "_")
    path = os.path.join(
        cache_dir,
        f"{safe_ds}_{split_name}_seq{conceptizer.seq_len}_tok{conceptizer.chunk_tok_len}.pt"
    )

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    rows = []
    skipped = 0
    print(f"[cache] building {dataset_name} / {split_name}")

    for ex in tqdm(split_data, desc=f"Conceptizing {dataset_name}-{split_name}"):
        try:
            stem, choice_texts, label = normalize_choices_field(ex)
            x = conceptizer.encode_choice_set(stem, choice_texts)
            choice_mask = torch.ones(x.size(0), dtype=torch.bool)

            rows.append({
                "x": x,
                "label": int(label),
                "choice_mask": choice_mask,
                "num_choices": int(x.size(0)),
            })
        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example in {dataset_name}-{split_name}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")
    return rows


class MCQFeatureDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx: int):
        r = self.rows[idx]
        return {
            "x": r["x"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "choice_mask": r["choice_mask"],
            "num_choices": r["num_choices"],
        }


def mcq_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    max_c = max(item["x"].size(0) for item in batch)
    t = batch[0]["x"].size(1)
    d = batch[0]["x"].size(2)
    b = len(batch)

    x = torch.zeros(b, max_c, t, d, dtype=torch.float32)
    choice_mask = torch.zeros(b, max_c, dtype=torch.bool)
    labels = torch.zeros(b, dtype=torch.long)

    for i, item in enumerate(batch):
        c = item["x"].size(0)
        x[i, :c] = item["x"]
        choice_mask[i, :c] = item["choice_mask"]
        labels[i] = item["label"]

    return {
        "x": x,
        "choice_mask": choice_mask,
        "labels": labels,
    }


# ============================================================
# MODEL
# ============================================================

class MCQHead(nn.Module):
    """
    Frozen HLCM + LayerNorm-stabilized classifier head.
    """
    def __init__(self, model: HyperbolicLCM, dropout: float = 0.5):
        super().__init__()
        self.model = model
        dim = model.layers[0].attn.embed_dim

        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dim, 1)

        nn.init.normal_(self.classifier.weight, mean=0.0, std=0.01)
        nn.init.zeros_(self.classifier.bias)

    def forward(
        self,
        x: torch.Tensor,
        choice_mask: torch.Tensor,
        mu: Optional[torch.Tensor] = None,
        sigma: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        b, c, t, din = x.shape
        x = x.reshape(b * c, t, din)

        if mu is not None and sigma is not None:
            x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

        with torch.no_grad():
            h = self.model(x)

        last_h = h[:, -1, :]
        last_tan = self.model.manifold.logmap0(last_h)
        last_tan = self.norm(last_tan)
        last_tan = self.dropout(last_tan)

        logits = self.classifier(last_tan).view(b, c)
        logits = logits.masked_fill(~choice_mask, -1e9)
        return logits


def build_hlcm_from_cfg(cfg: FinetuneConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: FinetuneConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"[load] loaded HLCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    model.eval()
    for p in model.parameters():
        p.requires_grad = False

    return model


# ============================================================
# OPTIM / SCHED
# ============================================================

class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps: int, total_steps: int, min_lr_scale: float = 0.1):
        self.optimizer = optimizer
        self.warmup_steps = max(0, int(warmup_steps))
        self.total_steps = max(1, int(total_steps))
        self.min_lr_scale = float(min_lr_scale)
        self.step_num = 0
        self.base_lrs = [pg["lr"] for pg in optimizer.param_groups]

    def step(self):
        self.step_num += 1
        s = self.step_num

        if s <= self.warmup_steps and self.warmup_steps > 0:
            scale = s / self.warmup_steps
        else:
            denom = max(1, self.total_steps - self.warmup_steps)
            p = min(1.0, max(0.0, (s - self.warmup_steps) / denom))
            cosine = 0.5 * (1.0 + math.cos(math.pi * p))
            scale = self.min_lr_scale + (1.0 - self.min_lr_scale) * cosine

        for lr0, pg in zip(self.base_lrs, self.optimizer.param_groups):
            pg["lr"] = lr0 * scale


# ============================================================
# TRAIN / EVAL
# ============================================================

@torch.no_grad()
def evaluate(
    model: MCQHead,
    loader: DataLoader,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> Dict[str, float]:
    model.eval()

    total_loss = 0.0
    total = 0
    correct = 0

    for batch in loader:
        x = batch["x"].to(device, non_blocking=True)
        choice_mask = batch["choice_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        logits = model(x, choice_mask, mu=mu, sigma=sigma)
        loss = masked_choice_cross_entropy(
            logits=logits,
            labels=labels,
            choice_mask=choice_mask,
            label_smoothing=0.0,
        )

        total_loss += float(loss.item()) * x.size(0)
        preds = logits.argmax(dim=-1)
        correct += int((preds == labels).sum().item())
        total += int(x.size(0))

    model.train()
    return {
        "loss": total_loss / max(total, 1),
        "acc": correct / max(total, 1),
    }


def train_one_dataset(dataset_name: str, cfg: FinetuneConfig):
    print(f"\n==================== {dataset_name} ====================")
    device = pick_device(cfg.prefer_gpu_index)
    set_seed(cfg.seed)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    out_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    ensure_dir(out_dir)
    ensure_dir(cfg.cache_dir)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=device,
    )

    raw_ds = load_raw_mcq_dataset(dataset_name)

    train_rows = build_or_load_cached_split(
        dataset_name, "train", raw_ds["train"], conceptizer, cfg.cache_dir
    )
    val_rows = build_or_load_cached_split(
        dataset_name, "validation", raw_ds["validation"], conceptizer, cfg.cache_dir
    )
    test_rows = build_or_load_cached_split(
        dataset_name, "test", raw_ds["test"], conceptizer, cfg.cache_dir
    )

    if len(train_rows) == 0:
        raise RuntimeError(f"No usable training rows for {dataset_name}")
    if len(val_rows) == 0:
        raise RuntimeError(f"No usable validation rows for {dataset_name}")

    train_ds = MCQFeatureDataset(train_rows)
    val_ds = MCQFeatureDataset(val_rows)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    test_loader = None
    if len(test_rows) > 0:
        test_ds = MCQFeatureDataset(test_rows)
        test_loader = DataLoader(
            test_ds,
            batch_size=cfg.eval_batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=(device.type == "cuda"),
            collate_fn=mcq_collate,
            drop_last=False,
        )

    hlcm = load_pretrained_hlcm(cfg, device)
    model = MCQHead(hlcm, dropout=cfg.head_dropout).to(device)

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        [{"params": trainable_params, "lr": cfg.head_lr, "weight_decay": cfg.weight_decay}]
    )

    steps_per_epoch = math.ceil(len(train_loader) / max(cfg.grad_accum_steps, 1))
    total_steps = max(1, cfg.epochs * steps_per_epoch)
    warmup_steps = int(cfg.warmup_ratio * total_steps)
    scheduler = WarmupCosineScheduler(
        optimizer,
        warmup_steps=warmup_steps,
        total_steps=total_steps
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"
    amp_dtype = torch.bfloat16

    history = []
    best_val_acc = -1.0
    best_path = os.path.join(out_dir, "best_mcq.pt")

    start_time = time.time()

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        running_loss = 0.0
        seen = 0

        pbar = tqdm(train_loader, desc=f"Train {dataset_name} epoch {epoch}/{cfg.epochs}", mininterval=1.0)

        for step, batch in enumerate(pbar, start=1):
            x = batch["x"].to(device, non_blocking=True)
            choice_mask = batch["choice_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=amp_dtype):
                    logits = model(x, choice_mask, mu=mu, sigma=sigma)
                    loss = masked_choice_cross_entropy(
                        logits=logits,
                        labels=labels,
                        choice_mask=choice_mask,
                        label_smoothing=float(cfg.label_smoothing),
                    )
            else:
                logits = model(x, choice_mask, mu=mu, sigma=sigma)
                loss = masked_choice_cross_entropy(
                    logits=logits,
                    labels=labels,
                    choice_mask=choice_mask,
                    label_smoothing=float(cfg.label_smoothing),
                )

            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True)
                continue

            (loss / cfg.grad_accum_steps).backward()

            running_loss += float(loss.item()) * x.size(0)
            seen += int(x.size(0))

            if step % cfg.grad_accum_steps == 0 or step == len(train_loader):
                torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            pbar.set_postfix(loss=f"{running_loss / max(seen, 1):.4f}")

        train_loss = running_loss / max(seen, 1)
        val_metrics = evaluate(model, val_loader, device, mu, sigma)

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["acc"],
        }

        if test_loader is not None:
            test_metrics = evaluate(model, test_loader, device, mu, sigma)
            row["test_loss"] = test_metrics["loss"]
            row["test_acc"] = test_metrics["acc"]
        else:
            test_metrics = None

        history.append(row)

        print(
            f"[epoch {epoch}] "
            f"train_loss={train_loss:.4f} "
            f"val_loss={val_metrics['loss']:.4f} "
            f"val_acc={val_metrics['acc']:.4f}"
            + (f" test_acc={test_metrics['acc']:.4f}" if test_metrics is not None else "")
        )

        if val_metrics["acc"] > best_val_acc:
            best_val_acc = float(val_metrics["acc"])
            torch.save(
                {
                    "cfg": asdict(cfg),
                    "dataset_name": dataset_name,
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "hlcm_state": model.model.state_dict(),
                    "best_val_acc": best_val_acc,
                    "history": history,
                },
                best_path,
            )
            print(f"[save] best checkpoint -> {best_path}")

    wall = time.time() - start_time

    best_obj = torch.load(best_path, map_location=device)
    model.load_state_dict(best_obj["model_state"], strict=True)

    final_val = evaluate(model, val_loader, device, mu, sigma)
    final_test = evaluate(model, test_loader, device, mu, sigma) if test_loader is not None else None

    summary = {
        "dataset_name": dataset_name,
        "best_val_acc": float(final_val["acc"]),
        "best_val_loss": float(final_val["loss"]),
        "test_acc": None if final_test is None else float(final_test["acc"]),
        "test_loss": None if final_test is None else float(final_test["loss"]),
        "epochs": cfg.epochs,
        "wall_time_sec": float(wall),
        "wall_time_hms": fmt_hms(wall),
        "num_train_examples": len(train_ds),
        "num_val_examples": len(val_ds),
        "num_test_examples": len(test_rows),
        "history": history,
    }

    with open(os.path.join(out_dir, "summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print(f"[done] {dataset_name}")
    print(json.dumps(summary, indent=2))

    return summary


# ============================================================
# MAIN
# ============================================================

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--ckpt_path", type=str, default="runs/hyperbolic_cluster/checkpoints/ckpt_best.pt")
    p.add_argument("--normalizer_path", type=str, default="normalizer.pt")
    p.add_argument("--out_dir", type=str, default="runs/mcq_hlcm_openbookqa")
    p.add_argument("--cache_dir", type=str, default="mcq_cache")

    p.add_argument("--epochs", type=int, default=3)
    p.add_argument("--train_batch_size", type=int, default=4)
    p.add_argument("--eval_batch_size", type=int, default=8)
    p.add_argument("--grad_accum_steps", type=int, default=1)

    p.add_argument("--lr", type=float, default=0.0)
    p.add_argument("--head_lr", type=float, default=3e-5)
    p.add_argument("--weight_decay", type=float, default=0.01)
    p.add_argument("--warmup_ratio", type=float, default=0.06)
    p.add_argument("--max_grad_norm", type=float, default=1.0)

    p.add_argument("--seq_len", type=int, default=8)
    p.add_argument("--chunk_tok_len", type=int, default=256)
    p.add_argument("--encoder_batch_size", type=int, default=64)

    p.add_argument("--prefer_gpu_index", type=int, default=0)
    p.add_argument("--seed", type=int, default=42)

    p.add_argument("--label_smoothing", type=float, default=0.10)

    if "ipykernel" in sys.modules:
        return p.parse_args(args=[])

    return p.parse_args()


def main():
    args = parse_args()

    cfg = FinetuneConfig(
        ckpt_path=args.ckpt_path,
        normalizer_path=args.normalizer_path,
        out_dir=args.out_dir,
        cache_dir=args.cache_dir,
        epochs=args.epochs,
        train_batch_size=args.train_batch_size,
        eval_batch_size=args.eval_batch_size,
        grad_accum_steps=args.grad_accum_steps,
        lr=args.lr,
        head_lr=args.head_lr,
        weight_decay=args.weight_decay,
        warmup_ratio=args.warmup_ratio,
        max_grad_norm=args.max_grad_norm,
        seq_len=args.seq_len,
        chunk_tok_len=args.chunk_tok_len,
        encoder_batch_size=args.encoder_batch_size,
        prefer_gpu_index=args.prefer_gpu_index,
        seed=args.seed,
        label_smoothing=args.label_smoothing,
        freeze_hlcm=True,
    )

    ensure_dir(cfg.out_dir)

    all_summaries = {}
    for ds_name in cfg.datasets_to_run:
        summary = train_one_dataset(ds_name, cfg)
        all_summaries[ds_name] = summary

    with open(os.path.join(cfg.out_dir, "all_results.json"), "w") as f:
        json.dump(all_summaries, f, indent=2)

    print("\n==================== ALL DONE ====================")
    print(json.dumps(all_summaries, indent=2))


if __name__ == "__main__":
    main()


==================== OpenBookQA ====================


/home/user/anaconda3/envs/raat/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


[cache] building OpenBookQA / train


Conceptizing OpenBookQA-train: 100%|█████████████████████████████████████████████████████████| 4957/4957 [01:38<00:00, 50.38it/s]


[cache] saved mcq_cache/OpenBookQA_train_seq8_tok256.pt (4957 examples, skipped=0)
[cache] building OpenBookQA / validation


...

[epoch 1] train_loss=1.6084 val_loss=1.3852 val_acc=0.3040 test_acc=0.3120
[save] best checkpoint -> runs/mcq_hlcm_openbookqa/OpenBookQA/best_mcq.pt


Train OpenBookQA epoch 2/3: 100%|███████████████████████████████████████████████| 1240/1240 [01:29<00:00, 13.81it/s, loss=1.5858]


[epoch 2] train_loss=1.5858 val_loss=1.3848 val_acc=0.3100 test_acc=0.3160
[save] best checkpoint -> runs/mcq_hlcm_openbookqa/OpenBookQA/best_mcq.pt


Train OpenBookQA epoch 3/3: 100%|███████████████████████████████████████████████| 1240/1240 [01:29<00:00, 13.83it/s, loss=1.5916]


[epoch 3] train_loss=1.5916 val_loss=1.3846 val_acc=0.3060 test_acc=0.3220
[done] OpenBookQA
{
  "dataset_name": "OpenBookQA",
  "best_val_acc": 0.31,
  "best_val_loss": 1.384789963722229,
  "test_acc": 0.316,
  "test_loss": 1.383479887008667,
  "epochs": 3,
  "wall_time_sec": 326.9669146537781,
  "wall_time_hms": "00:05:26",
  "num_train_examples": 4957,
  "num_val_examples": 500,
  "num_test_examples": 500,
  "history": [
    {
      "epoch": 1,
      "train_loss": 1.6084150200244134,
      "val_loss": 1.3851924781799316,
      "val_acc": 0.304,
      "test_loss": 1.3838309574127197,
      "test_acc": 0.312
    },
    {
      "epoch": 2,
      "train_loss": 1.5858492537903068,
      "val_loss": 1.384789963722229,
      "val_acc": 0.31,
      "test_loss": 1.383479887008667,
      "test_acc": 0.316
    },
    {
      "epoch": 3,
      "train_loss": 1.5915510063813967,
      "val_loss": 1.3846353244781495,
      "val_acc": 0.306,
      "test_loss": 1.383381546020508,
      "test_acc": 0

In [1]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import json
import time
import random
from dataclasses import dataclass
from typing import List, Dict, Any, Optional, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class InferenceConfig:
    dataset_name: str = "OpenBookQA"

    cache_dir: str = "mcq_cache"
    out_dir: str = "runs/mcq_hlcm_openbookqa"

    best_ckpt_path: str = "runs/mcq_hlcm_openbookqa/OpenBookQA/best_mcq.pt"
    normalizer_path: str = "normalizer.pt"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 64

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    head_dropout: float = 0.50

    eval_batch_size: int = 8
    num_workers: int = 4
    prefer_gpu_index: int = 0
    seed: int = 42

    split: str = "test"

    # If True, includes DeBERTa conceptization time when cache is missing.
    # If cache exists, inference time measures only H-LCM + MCQ head forward pass.
    build_cache_if_missing: bool = True


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds: float) -> str:
    seconds = max(float(seconds), 0.0)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def answerkey_to_index(answer_key: str, num_choices: int) -> int:
    answer_key = str(answer_key).strip()
    alpha = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
    numeric = {"1": 0, "2": 1, "3": 2, "4": 3, "5": 4}

    if answer_key in alpha:
        idx = alpha[answer_key]
    elif answer_key in numeric:
        idx = numeric[answer_key]
    else:
        raise ValueError(f"Unknown answerKey: {answer_key}")

    if idx >= num_choices:
        raise ValueError(
            f"answerKey={answer_key} -> idx={idx}, but num_choices={num_choices}"
        )

    return idx


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        print("[normalizer] not found; using None")
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)

    print(f"[normalizer] loaded from {normalizer_path}")
    return mu, sigma


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.encoder = AutoModel.from_pretrained(model_name).to(device)
        self.encoder.eval()

        for p in self.encoder.parameters():
            p.requires_grad = False

        self.embed_dim = int(self.encoder.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.embed_dim, dtype=torch.float32)

        inputs = self.tokenizer(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device, non_blocking=True) for k, v in inputs.items()}

        if self.device.type == "cuda":
            dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
            with amp.autocast(device_type="cuda", enabled=True, dtype=dtype):
                out = self.encoder(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.encoder(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _text_to_chunk_texts(self, text: str) -> List[str]:
        ids = self.tokenizer(text, add_special_tokens=False)["input_ids"]

        if len(ids) == 0:
            return []

        chunk_texts = []
        for i in range(0, len(ids), self.chunk_tok_len):
            chunk_ids = ids[i:i + self.chunk_tok_len]
            chunk_text = self.tokenizer.decode(
                chunk_ids,
                clean_up_tokenization_spaces=True,
            )
            chunk_texts.append(chunk_text)

        return chunk_texts

    def encode_text(self, text: str) -> torch.Tensor:
        chunk_texts = self._text_to_chunk_texts(text)

        if len(chunk_texts) == 0:
            return torch.zeros(self.seq_len, self.embed_dim, dtype=torch.float32)

        chunk_texts = chunk_texts[:self.seq_len]

        out = []
        for i in range(0, len(chunk_texts), self.batch_size):
            batch = chunk_texts[i:i + self.batch_size]
            out.append(self._embed_chunk_texts(batch))

        embs = torch.cat(out, dim=0)

        if embs.size(0) < self.seq_len:
            pad = torch.zeros(
                self.seq_len - embs.size(0),
                embs.size(1),
                dtype=embs.dtype,
            )
            embs = torch.cat([embs, pad], dim=0)

        return embs[:self.seq_len]

    def encode_choice_set(self, question_stem: str, choice_texts: List[str]) -> torch.Tensor:
        feats = []

        for ch in choice_texts:
            text = f"Question: {question_stem}\nAnswer Choice: {ch}"
            feats.append(self.encode_text(text))

        return torch.stack(feats, dim=0)


# ============================================================
# DATA
# ============================================================

def load_raw_mcq_dataset(dataset_name: str):
    if dataset_name == "OpenBookQA":
        return load_dataset("allenai/openbookqa")

    raise ValueError(f"Unknown dataset: {dataset_name}")


def normalize_choices_field(example: Dict[str, Any]) -> Tuple[str, List[str], int]:
    if "question_stem" not in example:
        raise KeyError("question_stem")

    stem = str(example["question_stem"])

    if "choices" not in example:
        raise KeyError("choices")

    choices = example["choices"]

    if not isinstance(choices, dict):
        raise TypeError(f"Expected choices dict, got {type(choices)}")

    if "text" not in choices:
        raise KeyError("choices.text")

    choice_texts = [str(x) for x in choices["text"]]

    if "answerKey" not in example:
        raise KeyError("answerKey")

    label = answerkey_to_index(example["answerKey"], len(choice_texts))

    return stem, choice_texts, label


def cache_path_for_split(cfg: InferenceConfig, split_name: str) -> str:
    safe_ds = cfg.dataset_name.replace("/", "_")
    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_seq{cfg.seq_len}_tok{cfg.chunk_tok_len}.pt",
    )


def build_or_load_cached_split(
    cfg: InferenceConfig,
    split_name: str,
    split_data,
    conceptizer: Optional[DebertaConceptizer],
) -> List[Dict[str, Any]]:
    ensure_dir(cfg.cache_dir)

    path = cache_path_for_split(cfg, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path, map_location="cpu")

    if not cfg.build_cache_if_missing:
        raise FileNotFoundError(
            f"Cache file not found: {path}\n"
            f"Set build_cache_if_missing=True if you want to build it."
        )

    if conceptizer is None:
        raise RuntimeError("conceptizer is required to build cache")

    rows = []
    skipped = 0

    print(f"[cache] building {cfg.dataset_name} / {split_name}")

    for ex in tqdm(split_data, desc=f"Conceptizing {cfg.dataset_name}-{split_name}"):
        try:
            stem, choice_texts, label = normalize_choices_field(ex)
            x = conceptizer.encode_choice_set(stem, choice_texts)
            choice_mask = torch.ones(x.size(0), dtype=torch.bool)

            rows.append(
                {
                    "x": x,
                    "label": int(label),
                    "choice_mask": choice_mask,
                    "num_choices": int(x.size(0)),
                }
            )
        except Exception as e:
            skipped += 1
            print(
                f"[warn] skipped one example in {cfg.dataset_name}-{split_name}: "
                f"{type(e).__name__}: {e}"
            )

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")

    return rows


class MCQFeatureDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx: int):
        r = self.rows[idx]

        return {
            "x": r["x"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "choice_mask": r["choice_mask"],
            "num_choices": r["num_choices"],
        }


def mcq_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    max_c = max(item["x"].size(0) for item in batch)
    t = batch[0]["x"].size(1)
    d = batch[0]["x"].size(2)
    b = len(batch)

    x = torch.zeros(b, max_c, t, d, dtype=torch.float32)
    choice_mask = torch.zeros(b, max_c, dtype=torch.bool)
    labels = torch.zeros(b, dtype=torch.long)

    for i, item in enumerate(batch):
        c = item["x"].size(0)

        x[i, :c] = item["x"]
        choice_mask[i, :c] = item["choice_mask"]
        labels[i] = item["label"]

    return {
        "x": x,
        "choice_mask": choice_mask,
        "labels": labels,
    }


# ============================================================
# MODEL
# ============================================================

class MCQHead(nn.Module):
    def __init__(self, model: HyperbolicLCM, dropout: float = 0.5):
        super().__init__()

        self.model = model
        dim = model.layers[0].attn.embed_dim

        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dim, 1)

    def forward(
        self,
        x: torch.Tensor,
        choice_mask: torch.Tensor,
        mu: Optional[torch.Tensor] = None,
        sigma: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        b, c, t, din = x.shape
        x = x.reshape(b * c, t, din)

        if mu is not None and sigma is not None:
            x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

        with torch.no_grad():
            h = self.model(x)

        last_h = h[:, -1, :]
        last_tan = self.model.manifold.logmap0(last_h)

        last_tan = self.norm(last_tan)
        last_tan = self.dropout(last_tan)

        logits = self.classifier(last_tan).view(b, c)
        logits = logits.masked_fill(~choice_mask, -1e9)

        return logits


def build_hlcm_from_cfg(cfg: InferenceConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_inference_model(cfg: InferenceConfig, device: torch.device) -> MCQHead:
    hlcm = build_hlcm_from_cfg(cfg).to(device)
    model = MCQHead(hlcm, dropout=cfg.head_dropout).to(device)

    if os.path.exists(cfg.best_ckpt_path):
        print(f"[load] loading fine-tuned checkpoint: {cfg.best_ckpt_path}")

        obj = torch.load(cfg.best_ckpt_path, map_location=device)

        if "model_state" not in obj:
            raise KeyError("Checkpoint does not contain 'model_state'.")

        missing, unexpected = model.load_state_dict(obj["model_state"], strict=False)

        print(f"[load] missing keys: {len(missing)}")
        print(f"[load] unexpected keys: {len(unexpected)}")
        print("[load] fine-tuned MCQ checkpoint loaded")

    else:
        print("=" * 80)
        print("[warning] Fine-tuned best_mcq.pt was NOT found.")
        print("[warning] Running inference with randomly initialized MCQ head.")
        print("[warning] Inference time is valid, but accuracy is NOT meaningful.")
        print("=" * 80)

        # Optional: load only pretrained HLCM backbone if available
        pretrained_path = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"

        if os.path.exists(pretrained_path):
            obj = torch.load(pretrained_path, map_location="cpu")
            state = obj["model"] if "model" in obj else obj

            missing, unexpected = hlcm.load_state_dict(state, strict=False)

            print(f"[load] loaded pretrained HLCM backbone from {pretrained_path}")
            print(f"[load] HLCM missing keys: {len(missing)}")
            print(f"[load] HLCM unexpected keys: {len(unexpected)}")
        else:
            print("[warning] pretrained HLCM checkpoint also not found.")
            print("[warning] using fully random HLCM + random MCQ head.")

    model.eval()

    for p in model.parameters():
        p.requires_grad = False

    return model


# ============================================================
# INFERENCE + TIMING
# ============================================================

@torch.no_grad()
def run_inference_with_time(
    model: MCQHead,
    loader: DataLoader,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    save_path: Optional[str] = None,
) -> Dict[str, Any]:
    model.eval()

    total = 0
    correct = 0
    predictions = []

    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    for batch in tqdm(loader, desc="Inference"):
        x = batch["x"].to(device, non_blocking=True)
        choice_mask = batch["choice_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        logits = model(x, choice_mask, mu=mu, sigma=sigma)
        preds = logits.argmax(dim=-1)

        batch_correct = preds.eq(labels)

        for i in range(x.size(0)):
            predictions.append(
                {
                    "example_index": total + i,
                    "gold": int(labels[i].item()),
                    "pred": int(preds[i].item()),
                    "correct": int(batch_correct[i].item()),
                    "logits": [float(v) for v in logits[i].detach().cpu().tolist()],
                }
            )

        correct += int(batch_correct.sum().item())
        total += int(x.size(0))

    if device.type == "cuda":
        torch.cuda.synchronize()

    inference_time_sec = time.perf_counter() - start_time

    results = {
        "num_examples": total,
        "correct": correct,
        "accuracy": correct / max(total, 1),
        "accuracy_percent": 100.0 * correct / max(total, 1),
        "inference_time_sec": inference_time_sec,
        "inference_time_hms": fmt_hms(inference_time_sec),
        "time_per_example_sec": inference_time_sec / max(total, 1),
        "examples_per_second": total / max(inference_time_sec, 1e-9),
        "predictions": predictions,
    }

    if save_path is not None:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=2)

        print(f"[save] inference results -> {save_path}")

    return results


# ============================================================
# MAIN NOTEBOOK FUNCTION
# ============================================================

def inference_only_openbookqa(cfg: InferenceConfig):
    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print(f"[device] {device}")

    ensure_dir(cfg.cache_dir)

    dataset_out_dir = os.path.join(cfg.out_dir, cfg.dataset_name.replace("/", "_"))
    ensure_dir(dataset_out_dir)

    raw_ds = load_raw_mcq_dataset(cfg.dataset_name)

    conceptizer = None
    cache_path = cache_path_for_split(cfg, cfg.split)

    cache_build_time_sec = 0.0

    if not os.path.exists(cache_path):
        print(f"[cache] not found: {cache_path}")

        if cfg.build_cache_if_missing:
            conceptizer = DebertaConceptizer(
                model_name=cfg.encoder_name,
                chunk_tok_len=cfg.chunk_tok_len,
                seq_len=cfg.seq_len,
                batch_size=cfg.encoder_batch_size,
                device=device,
            )

            if device.type == "cuda":
                torch.cuda.synchronize()

            cache_start = time.perf_counter()

            rows = build_or_load_cached_split(
                cfg=cfg,
                split_name=cfg.split,
                split_data=raw_ds[cfg.split],
                conceptizer=conceptizer,
            )

            if device.type == "cuda":
                torch.cuda.synchronize()

            cache_build_time_sec = time.perf_counter() - cache_start
        else:
            raise FileNotFoundError(cache_path)
    else:
        rows = build_or_load_cached_split(
            cfg=cfg,
            split_name=cfg.split,
            split_data=raw_ds[cfg.split],
            conceptizer=None,
        )

    print(f"[data] split={cfg.split}, examples={len(rows)}")

    ds = MCQFeatureDataset(rows)

    loader = DataLoader(
        ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    model = load_inference_model(cfg, device)
    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    save_path = os.path.join(
        dataset_out_dir,
        f"inference_only_{cfg.split}_results.json",
    )

    inference_results = run_inference_with_time(
        model=model,
        loader=loader,
        device=device,
        mu=mu,
        sigma=sigma,
        save_path=save_path,
    )

    summary = {
        "dataset_name": cfg.dataset_name,
        "split": cfg.split,
        "checkpoint": cfg.best_ckpt_path,
        "cache_file": cache_path,
        "num_examples": inference_results["num_examples"],
        "correct": inference_results["correct"],
        "accuracy": inference_results["accuracy"],
        "accuracy_percent": inference_results["accuracy_percent"],
        "cache_build_time_sec": cache_build_time_sec,
        "cache_build_time_hms": fmt_hms(cache_build_time_sec),
        "inference_time_sec": inference_results["inference_time_sec"],
        "inference_time_hms": inference_results["inference_time_hms"],
        "time_per_example_sec": inference_results["time_per_example_sec"],
        "examples_per_second": inference_results["examples_per_second"],
    }

    summary_path = os.path.join(
        dataset_out_dir,
        f"inference_only_{cfg.split}_summary.json",
    )

    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)

    print("\n==================== INFERENCE DONE ====================")
    print(json.dumps(summary, indent=2))
    print(f"[summary saved] {summary_path}")

    return summary, inference_results

In [2]:
cfg = InferenceConfig(
    dataset_name="OpenBookQA",

    best_ckpt_path="runs/mcq_hlcm_openbookqa/OpenBookQA/best_mcq.pt",
    normalizer_path="normalizer.pt",

    cache_dir="mcq_cache",
    out_dir="runs/mcq_hlcm_openbookqa",

    split="test",
    eval_batch_size=8,
    prefer_gpu_index=0,

    build_cache_if_missing=True,
)

summary, inference_results = inference_only_openbookqa(cfg)

[device] cuda:0


Using the latest cached version of the dataset since allenai/openbookqa couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'main' at /home/user/.cache/huggingface/datasets/allenai___openbookqa/main/0.0.0/388097ea7776314e93a529163e0fea805b8a6454 (last modified on Wed Mar 18 08:37:00 2026).


[cache] loading mcq_cache/OpenBookQA_test_seq8_tok256.pt
[data] split=test, examples=500
[warning] Fine-tuned best_mcq.pt was NOT found.
[warning] Running inference with randomly initialized MCQ head.
[warning] Inference time is valid, but accuracy is NOT meaningful.
[load] loaded pretrained HLCM backbone from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] HLCM missing keys: 0
[load] HLCM unexpected keys: 0
[normalizer] loaded from normalizer.pt


Inference: 100%|████████████████████████████████████████████████████| 63/63 [00:11<00:00,  5.63it/s]

[save] inference results -> runs/mcq_hlcm_openbookqa/OpenBookQA/inference_only_test_results.json

==================== INFERENCE DONE ====================
{
  "dataset_name": "OpenBookQA",
  "split": "test",
  "checkpoint": "runs/mcq_hlcm_openbookqa/OpenBookQA/best_mcq.pt",
  "cache_file": "mcq_cache/OpenBookQA_test_seq8_tok256.pt",
  "num_examples": 500,
  "correct": 130,
  "accuracy": 0.26,
  "accuracy_percent": 26.0,
  "cache_build_time_sec": 0.0,
  "cache_build_time_hms": "00:00:00",
  "inference_time_sec": 11.271639277692884,
  "inference_time_hms": "00:00:11",
  "time_per_example_sec": 0.02254327855538577,
  "examples_per_second": 44.35912006069286
}
[summary saved] runs/mcq_hlcm_openbookqa/OpenBookQA/inference_only_test_summary.json


In [3]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import math
import random
from typing import List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

from h_lcm1 import HyperbolicLCM


# ============================================================
# DEVICE
# ============================================================

assert torch.cuda.is_available()
torch.cuda.set_device(0)
DEVICE = torch.device("cuda:0")

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


# ============================================================
# CONFIG (DEGRADE SETTINGS)
# ============================================================

PRETRAIN_CKPT = "/home/user/twovolume/Nisha/hlcm/runs/hyperbolic_cluster_only_mixed/checkpoints/ckpt_best.pt"

MODEL_NAME = "microsoft/deberta-v3-small"
CHUNK_TOK_LEN = 256

# --- Degrade option: reduce sequence length (less info) ---
SEQ_LEN = 2          # <<< was 8; set 2 to make it worse (you can try 1 too)

# --- Freeze policy: train as little as possible ---
TRAIN_LAST_N_LAYERS = 0   # <<< 0 means do NOT train any transformer layers
TRAIN_INPUT_PROJ = False  # <<< False makes it worse (head-only)

# --- Degrade schedule ---
NUM_EPOCHS = 1            # <<< fewer epochs makes it worse
TRAIN_BS = 2
EVAL_BS = 4
GRAD_ACCUM = 8

# --- Degrade optimization ---
LR = 5e-5                 # <<< smaller LR -> weaker adaptation
WEIGHT_DECAY = 0.1        # <<< stronger regularization -> worse
GRAD_CLIP_NORM = 1.0

# --- Degrade head ---
HEAD_DROPOUT = 0.6        # <<< high dropout -> worse

# --- Degrade data signal ---
EMB_NOISE_STD = 0.25      # <<< add noise to DeBERTa CLS embeddings (major drop)
# Suggested: 0.05 (mild), 0.15 (medium), 0.25 (strong), 0.5 (very strong)

# --- Optional: train on small subset to reduce learning ---
USE_TRAIN_SUBSET = True
TRAIN_SUBSET_N = 2000     # <<< train on only first N examples (smaller -> worse)

RUN_DIR = "./runs_finetune"
os.makedirs(RUN_DIR, exist_ok=True)


# ============================================================
# ARCH INFERENCE FROM CHECKPOINT
# ============================================================

def infer_arch_from_state_dict(state: dict):
    model_dim, in_dim = state["input_proj.weight"].shape
    layer_ids = set()
    for k in state.keys():
        if k.startswith("layers."):
            layer_ids.add(int(k.split(".")[1]))
    num_layers = max(layer_ids) + 1
    return {"in_dim": in_dim, "model_dim": model_dim, "num_layers": num_layers}


# ============================================================
# EMBEDDER (CPU) - returns normal tensors
# ============================================================

class ChunkCLSEmbedder:
    def __init__(self, model_name, chunk_len, seq_len):
        self.chunk_len = int(chunk_len)
        self.seq_len = int(seq_len)

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        if self.tok.pad_token_id is None:
            self.tok.pad_token = self.tok.eos_token

        self.enc = AutoModel.from_pretrained(model_name).cpu()
        self.enc.eval()
        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden = int(self.enc.config.hidden_size)

    @torch.no_grad()  # stable + normal tensors
    def embed_texts(self, texts: List[str]) -> Tuple[torch.Tensor, torch.Tensor]:
        all_ids = [self.tok(t, add_special_tokens=False)["input_ids"] for t in texts]

        chunk_input_ids = []
        chunk_attn = []
        owner = []

        for i, ids in enumerate(all_ids):
            if len(ids) == 0:
                continue

            j = 0
            chunks = []
            while j < len(ids) and len(chunks) < self.seq_len:
                chunk = ids[j:j + self.chunk_len]
                if len(chunk) == 0:
                    break
                chunks.append(chunk)
                j += self.chunk_len

            for c in chunks:
                pad_len = self.chunk_len - len(c)
                inp = c + [self.tok.pad_token_id] * pad_len
                attn = [1] * len(c) + [0] * pad_len
                chunk_input_ids.append(inp)
                chunk_attn.append(attn)
                owner.append(i)

        N = len(texts)
        embs = torch.zeros(N, self.seq_len, self.hidden, dtype=torch.float32, device=DEVICE)
        mask = torch.zeros(N, self.seq_len, dtype=torch.bool, device=DEVICE)

        if len(chunk_input_ids) == 0:
            return embs.clone(), mask.clone()

        input_ids = torch.tensor(chunk_input_ids, dtype=torch.long, device="cpu")
        attention_mask = torch.tensor(chunk_attn, dtype=torch.long, device="cpu")

        cls = self.enc(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :].float()
        cls = cls.to(DEVICE, non_blocking=True)

        pos_ctr = [0] * N
        for idx, ex_i in enumerate(owner):
            p = pos_ctr[ex_i]
            if p < self.seq_len:
                embs[ex_i, p] = cls[idx]
                mask[ex_i, p] = True
                pos_ctr[ex_i] += 1

        return embs.clone(), mask.clone()


# ============================================================
# DATA
# ============================================================

def build_pair_text(q, c):
    return f"Question: {q}\nChoice: {c}"

def normalize_choices(obj):
    if isinstance(obj, dict):
        return list(zip(obj["label"], obj["text"]))
    return [(c["label"], c["text"]) for c in obj]

def label_to_index(ans, labels):
    return labels.index(ans) if ans in labels else 0


class MCQDataset(torch.utils.data.Dataset):
    def __init__(self, hf_split, task):
        self.ds = hf_split
        self.task = task

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        ex = self.ds[idx]
        q = ex["question"] if self.task.startswith("ARC") else ex["question_stem"]
        pairs = normalize_choices(ex["choices"])
        labels = [l for l, _ in pairs]
        texts = [build_pair_text(q, t) for _, t in pairs]
        y = label_to_index(ex["answerKey"], labels)
        return {"texts": texts, "label": y, "k": len(texts)}


def make_collate(embedder):
    def collate(batch):
        B = len(batch)
        maxK = max(b["k"] for b in batch)

        flat = []
        choice_mask = torch.zeros(B, maxK, dtype=torch.bool)
        labels = torch.tensor([b["label"] for b in batch], dtype=torch.long, device=DEVICE)

        for i, b in enumerate(batch):
            choice_mask[i, :b["k"]] = True
            flat.extend(b["texts"])
            flat.extend([""] * (maxK - b["k"]))

        embs, concept_mask = embedder.embed_texts(flat)  # [B*K,S,768] on cuda
        # --- DEGRADE: inject noise into embeddings ---
        if EMB_NOISE_STD and EMB_NOISE_STD > 0:
            embs = embs + float(EMB_NOISE_STD) * torch.randn_like(embs)

        embs = embs.view(B, maxK, SEQ_LEN, -1)
        concept_mask = concept_mask.view(B, maxK, SEQ_LEN)

        return {
            "choices_emb": embs,
            "concept_mask": concept_mask,
            "choice_mask": choice_mask.to(DEVICE),
            "label": labels,
        }
    return collate


# ============================================================
# HEAD
# ============================================================

class MCQHead(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.ln = nn.LayerNorm(dim)
        self.drop = nn.Dropout(HEAD_DROPOUT)
        self.fc = nn.Linear(dim, 1)

    def forward(self, x):
        return self.fc(self.drop(self.ln(x))).squeeze(-1)


# ============================================================
# FORWARD
# ============================================================

def forward_mcq(model, head, batch):
    x = batch["choices_emb"]
    cmask = batch["concept_mask"]
    chmask = batch["choice_mask"]
    y = batch["label"]

    B, K, S, D = x.shape
    x = x.view(B * K, S, D).float()
    m = cmask.view(B * K, S).float()

    out = model(x)
    out_tan = model.manifold.logmap0(out)

    denom = m.sum(dim=1, keepdim=True).clamp_min(1.0)
    pooled = (out_tan * m.unsqueeze(-1)).sum(dim=1) / denom

    logits = head(pooled).view(B, K)
    logits = logits.masked_fill(~chmask, -1e9)

    loss = F.cross_entropy(logits, y)
    acc = (logits.argmax(dim=1) == y).float().mean()
    return loss, acc


# ============================================================
# TRAIN
# ============================================================

def train_one(dataset_name, embedder):
    print(f"\n==== {dataset_name} ====")

    if dataset_name == "ARC-Easy":
        raw = load_dataset("allenai/ai2_arc", "ARC-Easy")
        task = "ARC-Easy"
    elif dataset_name == "ARC-Challenge":
        raw = load_dataset("allenai/ai2_arc", "ARC-Challenge")
        task = "ARC-Challenge"
    else:
        raw = load_dataset("allenai/openbookqa", "main")
        task = "OpenBookQA"

    train_ds = MCQDataset(raw["train"], task)
    val_ds = MCQDataset(raw["validation"], task)

    # --- DEGRADE: use only a small subset ---
    if USE_TRAIN_SUBSET:
        n = min(TRAIN_SUBSET_N, len(train_ds))
        train_ds = torch.utils.data.Subset(train_ds, list(range(n)))
        print(f"Using TRAIN SUBSET: {n}/{len(raw['train'])}")

    train_loader = DataLoader(
        train_ds, batch_size=TRAIN_BS, shuffle=True,
        collate_fn=make_collate(embedder), num_workers=0
    )
    val_loader = DataLoader(
        val_ds, batch_size=EVAL_BS, shuffle=False,
        collate_fn=make_collate(embedder), num_workers=0
    )

    # ===== LOAD CKPT =====
    ckpt = torch.load(PRETRAIN_CKPT, map_location="cpu")
    state = ckpt["model"]
    arch = infer_arch_from_state_dict(state)
    print("CKPT ARCH:", arch)

    model = HyperbolicLCM(
        in_dim=arch["in_dim"],
        model_dim=arch["model_dim"],
        num_heads=16,
        num_layers=arch["num_layers"],
        ffn_mult=4,
        dropout=0.1,
        manifold_c=0.05,
        init_scale=0.02,
        causal=False,
        input_scale=0.05,
        input_max_norm=1.0,
    )
    model.load_state_dict(state, strict=True)
    model = model.to(DEVICE)

    # infer hidden dim
    model.eval()
    with torch.no_grad():
        b = next(iter(val_loader))
        H = model(b["choices_emb"].view(-1, SEQ_LEN, arch["in_dim"])).shape[-1]
    model.train()

    head = MCQHead(H).to(DEVICE)

    # ===== FREEZE POLICY (DEGRADE) =====
    # Freeze everything by default (worst)
    for p in model.parameters():
        p.requires_grad = False

    # Optionally allow minimal adaptation (still often worse than full FT)
    if TRAIN_INPUT_PROJ:
        for p in model.input_proj.parameters():
            p.requires_grad = True

    if TRAIN_LAST_N_LAYERS > 0:
        for p in model.layers[-TRAIN_LAST_N_LAYERS:].parameters():
            p.requires_grad = True

    for p in head.parameters():
        p.requires_grad = True

    trainable = [p for p in list(model.parameters()) + list(head.parameters()) if p.requires_grad]
    total_params = sum(p.numel() for p in list(model.parameters()) + list(head.parameters()))
    trainable_params = sum(p.numel() for p in trainable)
    print("Trainable params:", trainable_params, f"({100.0*trainable_params/max(1,total_params):.4f}%)")

    optimizer = torch.optim.AdamW(trainable, lr=LR, weight_decay=WEIGHT_DECAY)

    # ===== TRAIN =====
    for epoch in range(NUM_EPOCHS):
        model.train()
        head.train()
        optimizer.zero_grad(set_to_none=True)

        for step, batch in enumerate(tqdm(train_loader)):
            loss, acc = forward_mcq(model, head, batch)
            if not torch.isfinite(loss):
                raise RuntimeError("Loss became NaN/Inf.")

            (loss / GRAD_ACCUM).backward()

            if (step + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(trainable, GRAD_CLIP_NORM)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

        # ===== EVAL =====
        model.eval()
        head.eval()
        total_acc = 0.0
        n = 0
        with torch.no_grad():
            for batch in val_loader:
                _, acc = forward_mcq(model, head, batch)
                bs = batch["label"].size(0)
                total_acc += acc.item() * bs
                n += bs

        print(f"{dataset_name} Epoch {epoch+1} VAL ACC:", total_acc / max(1, n))


# ============================================================
# MAIN
# ============================================================

embedder = ChunkCLSEmbedder(MODEL_NAME, CHUNK_TOK_LEN, SEQ_LEN)

for ds in ["ARC-Easy", "ARC-Challenge", "OpenBookQA"]:
    train_one(ds, embedder)

/home/user/anaconda3/envs/raat/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(



==== ARC-Easy ====
Using TRAIN SUBSET: 2000/2251
CKPT ARCH: {'in_dim': 768, 'model_dim': 4096, 'num_layers': 12}
Trainable params: 12289 (0.0005%)


100%|████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [07:49<00:00,  2.13it/s]


ARC-Easy Epoch 1 VAL ACC: 0.2578947368421053

==== ARC-Challenge ====
Using TRAIN SUBSET: 1119/1119
CKPT ARCH: {'in_dim': 768, 'model_dim': 4096, 'num_layers': 12}
Trainable params: 12289 (0.0005%)


100%|██████████████████████████████████████████████████████████████████████████████████████████| 560/560 [04:36<00:00,  2.02it/s]


ARC-Challenge Epoch 1 VAL ACC: 0.2775919734434938

==== OpenBookQA ====
Using TRAIN SUBSET: 2000/4957
CKPT ARCH: {'in_dim': 768, 'model_dim': 4096, 'num_layers': 12}
Trainable params: 12289 (0.0005%)


100%|████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [07:57<00:00,  2.09it/s]


OpenBookQA Epoch 1 VAL ACC: 0.296


In [1]:
import torch
import geoopt

print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("geoopt:", geoopt.__version__)

torch: 2.7.0+cu126
torch cuda: 12.6
geoopt: 0.5.0


In [3]:
import os, glob

RUN_DIR = "/home/user/twovolume/Nisha/hlcm/runs/hyperbolic_final3"
cands = []
for ext in ("*.pt", "*.pth", "*.bin", "*.ckpt"):
    cands += glob.glob(os.path.join(RUN_DIR, "**", ext), recursive=True)

print("Found checkpoint-like files:")
for p in sorted(cands):
    print(" -", p)

Found checkpoint-like files:
 - /home/user/twovolume/Nisha/hlcm/runs/hyperbolic_final3/checkpoints/ckpt_best.pt
 - /home/user/twovolume/Nisha/hlcm/runs/hyperbolic_final3/checkpoints/ckpt_step_004000.pt
 - /home/user/twovolume/Nisha/hlcm/runs/hyperbolic_final3/checkpoints/ckpt_step_004100.pt
 - /home/user/twovolume/Nisha/hlcm/runs/hyperbolic_final3/checkpoints/ckpt_step_004200.pt


In [4]:
import torch

ckpt = torch.load(
    "/home/user/twovolume/Nisha/hlcm/runs/hyperbolic_final3/checkpoints/ckpt_best.pt",
    map_location="cpu"
)

print(type(ckpt))
if isinstance(ckpt, dict):
    print("Top-level keys:", ckpt.keys())

<class 'dict'>
Top-level keys: dict_keys(['cfg', 'state', 'model', 'opt'])
